In [78]:
# Initialize graph

import geopandas as gpd
import networkx as nx
from contextlib import contextmanager
import time
import osmnx as ox
import numpy as np
import copy
import os
import json
from shapely.geometry import shape


@contextmanager
def perfTimer(action=""):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"Elapsed time to {action}: {round(end - start, 2)}s")

with perfTimer("read nodes and edges of mabalacat+angeles City"):
    nodes = gpd.read_file("graph/transit_graph/walking_graph/mabalacatAngelesWalking_nodes.geojson")
    edges = gpd.read_file("graph/transit_graph/walking_graph/mabalacatAngelesWalking_edges.geojson")

with perfTimer("Fix index of nodes and edges"):
    nodes = nodes.set_index("osmid")
    edges = edges.set_index(["u", "v", "key"])


with perfTimer("convert np arrays to lists"):
    for attr in edges.columns:
        new_values = []

        for val in edges[attr]:
            if isinstance(val, np.ndarray):
                new_values.append(val.tolist())
            else:
                new_values.append(val)

        edges[attr] = new_values

with perfTimer("initialize MultiDiGraph"):
    G = ox.graph_from_gdfs(nodes, edges)

with perfTimer("deepcopy MultiDiGraph to walking graph"):
    walkingG = copy.deepcopy(G)


with perfTimer("add tricycle subgraphs"):
    relativePath = "graph/transit_graph/trike_routes"

    for route in os.listdir(relativePath):

        for file in os.listdir(f"{relativePath}/{route}"):

            if file.endswith('_nodes.geojson'):
                with open(f"{relativePath}/{route}/{file}") as f:
                    nodesJson = json.load(f)

                for feature in nodesJson["features"]:
                    props = feature["properties"].copy()
                    geom = feature["geometry"]
                    node_id = props["osmid"]
                    lon, lat = geom["coordinates"]
                    props["x"] = lon
                    props["y"] = lat

                    G.add_node(node_id, **props)

            elif file.endswith('_edges.geojson'):
                with open(f"{relativePath}/{route}/{file}") as f:
                    edgesJson = json.load(f)

                for feature in edgesJson["features"]:
                    props = feature["properties"]
                    geom = feature["geometry"]

                    u = props.pop("u")
                    v = props.pop("v")

                    G.add_edge(
                        u,
                        v,
                        geometry=shape(geom),
                        **props
                    )
            else:
                print("Throw not correct format")


# Loops through every route json 2

relativePath = "graph/transit_graph/jeep_routes"

for route in os.listdir(relativePath):

    for file in os.listdir(f"{relativePath}/{route}"):

        if file.endswith('_nodes.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                nodesJson = json.load(f)

            for feature in nodesJson["features"]:
                props = feature["properties"].copy()
                geom = feature["geometry"]
                node_id = props["osmid"]
                lon, lat = geom["coordinates"]
                props["x"] = lon
                props["y"] = lat

                G.add_node(node_id, **props)

        elif file.endswith('_edges.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                edgesJson = json.load(f)

            for feature in edgesJson["features"]:
                props = feature["properties"]
                geom = feature["geometry"]

                u = props.pop("u")
                v = props.pop("v")

                G.add_edge(
                    u,
                    v,
                    geometry=shape(geom),
                    **props
                )
        else:
            print("Throw not correct format")

c:\Users\Adriane\Documents\bscs practice 2\sakenph-backend\venv\Lib\site-packages\geopandas\io\file.py:576: UserWarning: Could not parse column 'reversed' as JSON; leaving as string
  return pyogrio.read_dataframe(path_or_bytes, bbox=bbox, **kwargs)


Elapsed time to read nodes and edges of mabalacat+angeles City: 5.07s
Elapsed time to Fix index of nodes and edges: 0.02s
Elapsed time to convert np arrays to lists: 0.82s
Elapsed time to initialize MultiDiGraph: 1.12s
Elapsed time to deepcopy MultiDiGraph to walking graph: 3.32s
Elapsed time to add tricycle subgraphs: 0.46s


In [77]:
# Add jeep routes to json 2

import copy
import json
import os

nodeIdIndex = 0
edgeIdIndex = 0

def addJeepRoute(
        nodeList: dict[str, list[int]], 
        routeName: str
    ):
    '''
    Creates a node and edges file in GeoJSON format from a list of nodes that constitutes a path aligned to the jeepney route

    Throws an error if an osmid from the node list does not exist in the graph
    Throws an error if two adjacent nodes in the list do not have an edge connecting one another in the graph

    
    Parameters
    ----------
    nodeList
        It should be a list of nodes that constitutes a path aligned to the jeepney route.
        It will throw an error if the node_id in the graph does not exist, or if two adjacent
        nodes in the list dont have an edge connecting each other

    routeName
        Name of the jeepney route, preferably what the label on the side of the jeepney says

    Returns
    ----------
    None
    '''
    global nodeIdIndex; nodeIdIndex += 1
    global edgeIdIndex; nodeIdIndex += 1
    osmIdDict = {}

    # Tracks the ids of walking nodes' equivalent mode nodes


    # Reads the nodes, edges json of the original graph
    with open('graph/original_graph/mabalacatAngeles_nodes.geojson', 'r') as file:
        nodes_jsonData = json.load(file)
    with open('graph/original_graph/mabalacatAngeles_edges.geojson', 'r') as file:
        edges_jsonData = json.load(file)

    # Json file of the jeepney route nodes to write
    jeepRouteNodesJson = {
        "type": "FeatureCollection",
        "name": f"{routeName}_nodes",
        "crs": {
            "type": "name", 
            "properties": { 
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84" 
            } 
        },
        "features": [

        ]
    }

    # Json file of the jeepney route edges to write
    jeepRouteEdgesJson = {
        "type": "FeatureCollection",
        "name": f"{routeName}_edges",
        "crs": {
            "type": "name", 
            "properties": { 
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84" 
            } 
        },
        "features": [

        ]
    }

    
    # New osm_id formula (a * {corresponding osmid of walking node} + b)
    a = 105002; b = 67203817 
    createNewOsmId = lambda ogId: a * ogId + b
    
    nodesVisited = {}

    fullList = [
        {"bound": item["bound"], "id": node}
        for item in nodeList
        for node in item["nodeList"]
    ]

    # --------------------- FIRST NODE -------------------------#
    # Looks for the first node id from the route list 
    # inside the graph and gets data about its original node

    jeepNodeToAdd = None
    for i in nodes_jsonData['features']:
        if i['properties']['osmid'] == fullList[0]['id']:
            jeepNodeToAdd = copy.deepcopy(i)

            # Creates a unique osm_id for the node of the route
            # by this formula (105002 * node_id + 67203817)
            # newOsmId = createNewOsmId(nodeList[0])

            # Assigns id to new mode node
            newOsmId = nodeIdIndex
            osmIdDict[fullList[0]['id']] = newOsmId

            # Changes old osm_id to new one
            jeepNodeToAdd['properties']['osmid'] = newOsmId

            jeepNodeToAdd['properties']['mode'] = {
                "type": "jeep",
                "details": {
                    "name": routeName,
                    "bound": fullList[0]['bound']
                }
            }

            # Makes sure that newly added node is not duplicated in
            # case it is traversed again in the route
            nodesVisited[fullList[0]['id']] = copy.deepcopy(jeepNodeToAdd)
            break

    if (jeepNodeToAdd):
        jeepRouteNodesJson["features"].append(jeepNodeToAdd)
    else:
        raise KeyError(f"Node with id \"{fullList[0]['id']}\" is not in the graph.")

    # Outbound and inbound jeep node weights
    # 470 for inbound weight is designated to add penalty for boarding,
    # used to discourage the algorithm from picking short jeepney trips
    outJeepWeight = 10; inJeepWeight = 300

    # Function that creates a new feature for the edgesJson['features'] dictionary
    inOut_edgeFeature = lambda u, v, id, nodeFeatures, direction: {
        "type": "Feature",
        "properties": {
            "u": u,
            "v": v,
            "key": 0,
            "osmid": str(id),
            "length": 0.01,
            "weight": inJeepWeight if direction=='in' else outJeepWeight if direction=='out' else '0'
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']],
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']]
            ]
        }
    }

    # Add inbound jeep edge
    jeepRouteEdgesJson['features'].append(inOut_edgeFeature(fullList[0]['id'], newOsmId, edgeIdIndex, jeepNodeToAdd, 'in'))

    # Add outbound jeep edge
    jeepRouteEdgesJson['features'].append(inOut_edgeFeature(newOsmId, fullList[0]['id'], edgeIdIndex, jeepNodeToAdd, 'out'))

    # ---------------- SECOND TO THE LAST NODE -------------------#
    for m in range(1, len(fullList)):
        nodeIdIndex += 1; edgeIdIndex += 1
        jeepNodeToAdd = None

        # Check if node has not been visited already in the route
        if fullList[m]['id'] not in nodesVisited.keys():

            # Find node in the graph
            for i in nodes_jsonData['features']:

                # Check if node matches
                if i['properties']['osmid'] == fullList[m]['id']:

                    # Get node feature details
                    jeepNodeToAdd = copy.deepcopy(i)

                    newOsmId = nodeIdIndex
                    osmIdDict[fullList[m]['id']] = newOsmId

                    jeepNodeToAdd['properties']['osmid'] = newOsmId

                    # Add walking node and its features to visited list
                    nodesVisited[fullList[m]['id']] = copy.deepcopy(jeepNodeToAdd)
                    break

            if (jeepNodeToAdd):
                jeepNodeToAdd["properties"]["mode"] = {
                    "type": "jeep",
                    "details": {
                        "name": routeName,
                        "bound": fullList[m]['bound']
                    }
                }
                jeepRouteNodesJson["features"].append(jeepNodeToAdd)
            else:
                raise KeyError(f"Node with id \"{fullList[m]['id']}\" is not in the graph.")

            # Inbound jeep edge
            jeepRouteEdgesJson['features'].append(inOut_edgeFeature(fullList[m]['id'], newOsmId, edgeIdIndex, jeepNodeToAdd, 'in'))

            # Outbound jeep edge
            jeepRouteEdgesJson['features'].append(inOut_edgeFeature(newOsmId, fullList[m]['id'], edgeIdIndex, jeepNodeToAdd, 'out'))

        else:
            newOsmId = osmIdDict[fullList[m]['id']]

        edgeDetails = None

        for i in edges_jsonData['features']:
            if i['properties']['u'] == fullList[m-1]['id'] and i['properties']['v'] == fullList[m]['id'] and i['properties']['key'] == 0:
                edgeDetails = copy.deepcopy(i)
                break
        
        jeepSpeed = 3.89 # m/s
        if (edgeDetails):
            geometry = edgeDetails['geometry']
            length = edgeDetails['properties']['length']
            edgeIdIndex += 1

            # Calculated from length by speed which corresponds to how long
            # it takes to traverse an edge based on jeepney's average speed
            # in seconds
            weight = round(round(length, 2) / jeepSpeed, 2)

            jeepRouteEdgesJson['features'].append({
                "type": "Feature",
                "properties": {
                    "u": osmIdDict[fullList[m-1]['id']],
                    "v": newOsmId,
                    "key": 0,
                    "osmid": str(edgeIdIndex),
                    "length": length,
                    "weight": weight
                },
                "geometry": geometry
            })
        else:
            warningInput = input(
                f"Edge with u({fullList[m-1]['id']}) and v({fullList[m]['id']}) pair does not exist!\n" \
                "Are you sure you want to continue? Route may have disconnected paths.\n" \
                "Input \"yes\" to continue with exporting: "
            )

            if warningInput.lower() != 'yes':
                raise KeyError('Edge with corresponding (u,v) pair does not exist in the graph.')
            
    if not os.path.exists(f"graph/transit_graph/jeep_routes/{routeName}"):
        os.makedirs(f"graph/transit_graph/jeep_routes/{routeName}")

    with open(f'graph/transit_graph/jeep_routes/{routeName}/{routeName}_nodes.geojson', 'w') as file:
        json.dump(jeepRouteNodesJson, file, indent=1)

    with open(f'graph/transit_graph/jeep_routes/{routeName}/{routeName}_edges.geojson', 'w') as file:
        json.dump(jeepRouteEdgesJson, file, indent=1)

# trike = 6.48m/s
# jeep = 3.89m/s
# walk = 1.25m/s

purpleJeepList = [
    {"bound": "south", "nodeList": [3289714296, 3289714287, 3289714288, 3289714289, 5548939409, 2858136284, 12154752, 9174981631,
    7920874965, 9174981623, 9174978808, 9174978810, 854298207, 508035457, 12022633769, 12022633771,
    1843464661, 6471612671, 12272206774, 12272206781, 518296295, 7675114210, 7675097457, 8856752776,
    660578739, 1843464640, 660578733, 660578714, 283119889, 6266014252, 9175071247, 6266014256,
    6266014261, 13478908834, 314892198, 3192760788, 7690166269, 7170588670, 9456891490, 9456891488,
    3343162812, 3350956127, 3377833798, 7690178685, 3377833806, 3377833811, 3339677203, 7690166237,
    9176768661, 538127247, 9176771003, 9176770995, 1853859118, 7675733172, 6869893309, 311674612,
    13476114367, 9176915517, 1301475778, 9176915518, 6869976572, 279718156, 279718155, 529856382,
    4952801798, 457281472, 517417851, 279718148, 3556332238, 6831097023, 3393291352, 7676166990,
    1872304937, 9191121906]},
    {"bound": "north", "nodeList": [529853828, 5549726320, 7675801284, 8318725917, 854202919, 7675801282,
    6257635296, 2627302000, 10979944928, 7676009004, 13476558708, 2627302003, 2627302029, 854186888,
    9191058308, 12206389590, 2892712059, 12206389589, 2627301958, 9191058305, 1841740882, 514546844,
    846548468, 8304931722, 312297107, 312297509, 9191021570, 9191021573, 6828588336, 312294586, 13509227792,
    6870297706, 6355364996, 9183623297, 13509227798, 8302829218, 13509227786, 13509227795, 9183623311, 13509227789, 311674987, 312291693, 313910490,
    10113743412, 6828588389, 13511942695, 518261998, 8856524323, 7675733173, 312291686, 3339627326, 3339627327,
    3339627338, 3378951301, 7675735513, 3339627332, 312290705, 9183484508, 9183484509, 312290206,
    312290039, 311685317, 9183433986, 311675085, 3461062895, 1853898866, 457304698, 457304699,
    457304700, 457304687, 457304701, 313909603, 457304686, 313909477, 457304684, 312286063,
    10981690263, 457304685, 311685631, 311685572, 3339675390, 13595164001, 13595164003, 1853859132, 371318628, 371318598,
    6261997074, 6261997071, 9176751278, 371319548, 7690166226, 7690166243, 9176713770, 9176713731,
    311674860, 9839968425, 10761230866, 10761230868, 9839968433, 3339677215, 7141979357, 7141979356,
    7690166266, 7141979338, 10673981164, 8180600311, 13478908836, 3192760788, 314892198, 13478908834,
    6266014261, 6266014256, 9175071247, 6266014252, 283119889, 660578714, 660578733, 1843464640,
    660578739, 8856752776, 7675097457, 7675114210, 518296295, 12272206781, 12272206774, 6471612671,
    1843464661, 12022625863, 12022633772, 283119899, 9174978813, 9174978804, 9174978797, 7920872766, 485657698,
    5548939411, 3289714301, 3289714306, 1540323475, 3289714286, 3289714296, 1540323472, 3289717496,
    1540323469, 5548940641, 6814957201, 507044703, 8333736630, 1540323460, 10897890037, 1540323459,
    10897890066, 12696282413, 1540323454, 502842658, 1540323435, 11222235956, 1540323431, 11222235964]}
]

MabalacatDauAngelesSouth = [
    {"bound": "south", "nodeList": [
    1164921469, 338870504, 1746305271, 9281253746,
    6721960062, 9281253739, 10915569984, 1676875041, 10115868925,
    27170580, 1127578337, 1127578280, 1127578261, 9873662651,
    10089451380, 702805363, 373053399, 373053562, 7672709922,
    11953603894, 11953603901, 373050429, 6754430756, 373055864,
    9565026086, 9565026083, 10915569993, 10094865002, 373058167,
    664675389, 13523873250, 13523873242, 373057767, 373059487,
    6754527012, 619711918, 4583729567, 27170582, 373060607,
    1746305070, 9200214420, 9200214427, 6128693193, 6128693190,
    7675886492, 6128693197, 9200214479, 9200214470, 6128693194,
    6755439378, 9200214493, 9200214484, 9200214495, 8995666502,
    8995666494, 13508535492, 8584482239, 13508535486, 8584482242,
    373067218, 373066911, 9902614009, 10915511848, 10096730144,
    7675841321, 373066663, 660080446, 4417021930, 373072324,
    373073903, 660080430, 9200023907, 10915476579, 373071791,
    10100121266, 10915476592, 9200023909, 373157342, 6754485678,
    9196209156, 13511838165, 9196209149, 8855984724, 13512727130,
    1287153547, 373074467, 9914956813, 9188090059, 9188090051,
    10100109626, 10915479710, 4417018186, 10915479713, 667984263,
    9188090067, 13512505319, 10915479715, 9188090072, 660080329,
    8277757188, 9187885413, 8277757197, 6789418210, 7697805105,
    10911198636, 7697805113, 456858339, 9187898741, 10911198632,
    9187898743, 9187898752, 5431180046, 8276793868, 1355844593,
    8276793869, 12758814586, 7045813348, 7697797625, 5383866267,
    1717305193, 7294252054, 5383866270, 7294252048, 6265418912,
    6265418908, 7294252050, 6789112453, 1355844557, 660114998,
    522303093, 4588450120, 9169978795, 660115002, 9169978796,
    660114999, 9169978790, 317278751, 522302845, 9169978789,
    13164687872, 13164687877, 7678595739, 7678595730, 660114982,
    522302774, 8277920070, 9170072420, 9170056510, 9170056511,
    9170056497, 9170056488, 522302671, 9170056487, 317288869,
    9170056483, 9187628054, 8278178678, 13250223434, 7674980613,
    10111017985, 9187628055, 7357508289, 372577036, 522301886,
    372578357, 7357508288, 372546258, 538858115, 522298465,
    7675045550, 522298220, 10911031191, 522298412, 11084719234,
    11084719235, 283119886, 6266037530, 372443621, 360697784,
    372528399, 6732277546, 11070696329, 9187839297, 6732277551,
    6732277555, 6732277553, 11138324288, 6732277533, 11138333012,
    6732277535, 11138333010, 7673464378, 8682825680, 2748923904,
    8682825683, 8682825685, 13504279964, 7357315008, 13504279973,
    7357315004, 7357315018, 7357332269, 315735804, 508035441,
    314854754, 7695561764, 518296269, 6814692361, 3289714300,
    3289714306, 1540323475, 3289714286
]}]

greyJeep = [
    {"bound": "south", "nodeList": [
        10298120821, 10298112466, 7920872738, 8693517510, 1540323467, 10981724689, 1540323459, 10897890037, 1540323460,
        8333736630, 507044703, 6814957201, 5548940641, 1540323469, 3289717496, 1540323472, 3289714296, 3289714287, 3289714288, 3289714289, 5548939409, 2858136284, 12154752, 9174981631, 7920874965, 9174981623, 9174978808, 9174978810,
        854298207, 508035457, 12022633769, 12022633771,
        1843464661, 6471612671, 12272206774, 12272206781, 518296295, 7675114210, 7675097457, 8856752776, 660578739, 1843464640, 660578733, 660578714, 
        283119889, 6266014252, 9175071247, 6266014256, 6266014261, 13478908834, 314892198, 3192760788, 7690166269, 7170588670, 
        9456891490, 9456891488, 3343162812, 3350956127, 3377833798, 7690178685, 3377833806, 3377833811, 3339677203, 7690166237,
        9176768661, 538127247, 9176771003, 9176770995, 1853859118, 7675733172, 6869893309, 311674612, 13476114367, 9176915517,
        1301475778, 9176915518, 6869976572, 279718156, 279718155, 529856382, 4952801798, 457281472, 517417851, 279718148,
        3556332238, 6831097023, 3393291352, 7676166990, 1872304937, 9191121906]},
    {
        "bound": "north", 
        "nodeList": [
    529853828, 5549726320, 312296969, 7675835589, 
    3393291362, 1872304943, 9191109020, 3393291365, 6831097024, 3556332238, 279718148, 517417851, 457281472, 4952801798, 
    529856382, 279718155, 279718156, 6869976572, 9176915518, 1301475778, 9176915517, 13476114367, 311674612, 6869893309, 
    7675733172, 1853859118, 9176770995, 9176771003, 538127247, 9176768661, 7690166237, 3339677203, 538127234, 3377833803, 
    3377833810, 3377833805, 3377833800, 9839968433, 3339677215, 7141979357, 7141979356, 7690166266, 7141979338, 10673981164, 
    8180600311, 13478908836, 3192760788, 314892198, 13478908834, 6266014261, 6266014256, 9175071247, 6266014252, 283119889, 
    660578714, 660578733, 1843464640, 660578739, 8856752776, 7675097457, 7675114210, 518296295, 12272206781, 12272206774,
    6471612671, 1843464661, 12022625863, 12022633772, 283119899, 9174978813, 9174978804, 9174978797, 7920872766, 485657698,
    5548939411, 3289714301, 3289714306, 1540323475, 3289714286, 3289714296, 1540323472, 3289717496,
    1540323469, 5548940641, 6814957201, 507044703, 8333736630, 1540323460, 10897890037, 1540323459, 10981724689, 1540323467,
    8693517510, 7920872738, 10298112466, 10298120821
        ]
    }
]

greepJeep = [
    {"bound": "south", "nodeList": [538127234, 371320824, 9176713742, 311674860, 9176713731,
    9176713770, 7690166243, 7690166226, 371319548, 9176751278,
    6261997071, 6261997074, 371318598, 371318628, 1853859132,
    13595164003, 13595164001, 3339675390, 311685572, 311685631,
    457304685, 10981690263, 312286063, 457304684, 313909477,
    457304686, 313909603, 7198601340, 9461721535, 7198611426,
    9460738254, 9460738264, 313909934, 7198611429, 311675085,
    9183433986, 311685317, 312290039, 312290206, 9183484509,
    9183484508, 312290705, 3339627335, 3339627323, 9183498724,
    3339627340, 312291686, 7675733173, 8856524323, 518261998,
    13511942695, 6828588389, 10113743412, 313910490, 312291693, 311674987,
    13509227789, 9183623311, 13509227795, 13509227786, 8302829218,
    13509227798, 9183623297, 6355364996, 6870297706, 13509227792,
    312294586, 6828588336, 9191021573, 9191021570, 312297509,
    312297107, 11098875724, 846547074, 12206389584]},

    {"bound": "north", "nodeList": [854186888,
    2627302029, 2627302003, 13476558708, 7676009004, 10979944928,
    2627302000, 6257635296, 7675801282, 854202919, 8318725917,
    7675801284, 5549726320, 312296969, 7675835589, 3393291362,
    1872304943, 9191109020, 3393291365, 516063451, 311674704,
    311674706, 311674707, 311674708, 13478908825, 311674709,
    516063467, 516063463, 516063460, 876562484, 6869941657,
    1301475776, 13476114367, 311674612, 6869893309, 7675733172,
    1853859118, 9176770995, 9176771003, 538127247, 9176768661,
    7690166237, 3339677203]}
    
    
    
]

MabalacatDauAngelesNorth = [
    {"bound": "north", "nodeList": [11468813213, 518296269, 7695561764, 314854754, 508035441, 315735804, 7357332269, 7357315018, 7357315004, 13504279973, 7357315008, 13504279964, 8682825685, 8682825683,
    2748923904, 8682825681, 1842375233, 6732277534, 8682963010, 7675047513, 6727297893, 11138324289, 9188048889, 9188048896, 315737081, 8682986717, 8682963011, 6732280689,
    1703686178, 12093573, 6727297891, 6732133037, 9187839296, 6732277546, 372528399, 360697784, 372443621, 6266037530, 283119886, 11084719235, 11084719234, 522298412,
    10911031191, 522298220, 7675045550, 522298465, 538858115, 372546258, 7357508288, 372578357, 522301886, 372577036, 7357508289, 9187628055, 10111017985, 7674980613,
    13250223434, 8278178678, 9187628054, 10911041326, 10911041327, 10911041325, 10911041330, 9170056487, 522302671, 9170056488, 9170056497, 9170056511, 9170056510,
    9170072420, 8277920070, 522302774, 660114982, 7678595730, 7678595739, 13164687877, 13164687872, 9169978789, 522302845, 317278751, 9169978790, 660114999, 9169978796, 660115002, 
    9169978795, 4588450120, 522303093, 660114998, 1355844557, 6789112453, 7294252050, 6265418908, 6265418912, 7294252048, 5383866270, 7294252054, 1717305193,
    5383866267, 7697797625, 7045813348, 12758814586, 8276793869, 1355844593, 8276793868, 5431180046, 9187898752, 9187898743, 10911198632, 10911198627, 10911198629,
    10911198633, 10911198634, 7697805105, 6789418210, 8277757197, 9187885413, 8277757188, 660080329, 9188090072, 10915479715, 13512505319, 9188090067, 667984263, 10915479713, 
    4417018186, 10915479710, 10100109626, 9188090051, 9188090059, 9914956813, 373074467, 1287153547, 13512727130, 8855984724, 9196209149, 13511838165, 9196209156, 6754485678, 
    373157342, 9200023909, 10915476592, 10100121266, 373071791, 10915476579, 9200023907, 660080430, 373073903, 373072324, 4417021930, 660080446, 373066663, 7675841321, 
    10096730144, 10915511848, 10915511851, 10915511854, 373066911, 373067218, 8584482242, 13508535486, 8584482239, 13508535492, 8995666494, 8995666502, 9200214495, 9200214484, 
    9200214493, 6755439378, 6128693194, 9200214470, 9200214479, 6128693197, 7675886492, 6128693190, 6128693193, 9200214427, 9200214420, 1746305070, 373060607, 27170582, 
    4583729567, 619711918, 6754527012, 373059487, 373057767, 13523873242, 13523873250, 664675389, 373058167, 10094865002, 10915569993, 9565026083, 9565026086, 373055864, 
    6754430756, 373050429, 11953603901, 11953603894, 7672709922, 373053562, 373053399, 702805363, 10089451380, 9873662651, 1127578261, 1127578280, 1127578337, 27170580, 
    1127578267, 1676875041, 10915569984, 9281253739, 6721960062, 9281253746, 1746305271, 338870504, 1746305338]}
    
]

redJeepsouth = [ {
    "bound": "south", "nodeList": [7062880500, 8314381227, 7678863215, 7357543432, 467474580, 1437165101,
    467474577, 467474572, 7679028447, 7679028426, 7679034338, 7679028297,
    7679028416, 8333356187, 7679141060, 1842177240, 12851026707, 
    467474499, 685031756, 8295816443, 8333356184, 7687081341, 7687081340, 685031742,
    685031744, 467474485, 1553304335, 8333356152, 467474483, 13202221026,
    7687413736, 13520949339, 7687413732, 1611281040, 2995803925, 8157625650, 
    10116227380, 8856779963, 8133225075, 467474456, 1553282932, 1553283018,
    7687180596, 335928051, 7687180614, 13521125533, 13521125526, 9209304637,
    8133136347, 1127811263, 335927199, 518350444, 335927205,
      
    678600037, 335927206, 335928611, 9196185183, 259451787, 9196185178, 9196185200,
    9196185216, 678598607, 336118916, 336118905, 518325254, 5237457718,
    7198611444, 1553283109, 12926128516, 259451803, 1553282070, 8856675041,
    6266023669, 259451806, 259451808, 5381130548, 8678858237, 9208235025,
    825688443, 538130945, 8678858244, 8678819203, 8678819191, 8679100815,
    7729780957, 8246815001, 9208151485, 9208151473, 9208235047, 311301311,
    9208235048, 9208151478, 8679100809, 4952801814, 12206509006, 7678673473,
    6266019820, 529860579, 7811346409, 12850243790, 4952801810, 8288643656,
    11189379855, 11189379854, 10118844741, 10118844747, 13511181485, 
    5242249585, 7675127838, 538132372, 525432797, 7675617840, 8855903746,
    13478908830, 660220495, 10097880774, 10097880773, 538132328, 5367950553,
    5367950558, 5367950561, 538132310, 538132324, 538132317, 529856393,
    7629608890, 369889624, 529856390, 8302600019, 7629608888, 678927873,
    529856392, 678927833, 538127222, 369888874, 369889623, 6285526167,
    672571122, 679319652, 369889621, 516063457, 369889620, 457304240, 457281470]}]

redJeepNorth = [
    {"bound": "north", "nodeList": [457281470, 457304240, 369889620, 516063457, 369889621, 679319652, 672571122,
    6285526167, 369889623, 369888874, 538127222, 678927833, 529856392,
    678927873, 7629608888, 8302600019, 529856390, 369889624, 7629608890,
    529856393, 538132317, 538132324, 538132310, 5367950561, 5367950558,
    10761261204, 7675158173, 8855953254, 8855953257, 660220495, 13478908830,
    8855903746, 7675617840, 525432797, 538132372, 7675127838, 5242249585,
    13511181485, 10118844747, 10118844741, 11189379854, 11189379855,
    8288643656, 4952801810, 12850243790, 7811346409, 529860579, 
    6266019820, 7678673473, 12206509006, 4952801814, 8679100809,
    9208151478, 9208235048, 311301311, 9208235047, 9208151473, 9208151485,
    8246815001, 7729780957, 8679100815, 8678819191, 8678819203, 
    8678858244, 538130945, 825688443, 9208235025, 8678858237, 5381130548,
    259451808, 259451806, 6266023669, 8856675041, 1553282070, 259451803,
    12926128516, 1553283109, 259451796, 5237457718, 518325254, 336118905,
    336118916, 678598607, 9196185216, 9196185200, 9196185178, 259451787,
    9196185183, 335928611, 335927206, 678600037,

    335927205, 518350444, 335927199, 1127811263, 8133136347,
    9209304637, 13521125526, 13521125533, 7687180614, 335928051,
    7687180596, 1553283018, 1553282932, 467474456, 8133225075,
    8856779963, 10116227380, 8157625650, 2995803925, 1611281040,
    7687413732, 13520949339, 7687413736, 13202221026, 467474483,
    8333356152, 1553304335, 467474485, 685031744, 685031742,
    7687081340, 7687081341, 8333356184, 8295816443, 685031756,
    467474499, 12851026707, 1842177240, 7679141060, 8333356187,
    7679028416, 7679028297, 7679034338, 7679028426, 7679028447,
    467474572, 467474577, 1437165101, 467474580, 7357543432,
    7678863215, 8314381227, 7062880500]}
]

yellowJeepNorth = [
        {"bound": "north", "nodeList": [
            10879937416, 7676073006, 5634345007, 9919674471, 457276893,
            3339627286, 12235036169, 3339627283, 3339627298, 3339627297,
            3339587462, 9193923289, 313918603, 9193923301, 1872304768,
            9193923314, 313913653, 7675071639, 5367974293, 5367974291,
            1841746736, 1841757566, 7675766449, 1841758381, 846538010,
            7675766470, 13503867537, 1841740882,
            514546844, 846548468, 8304931722, 312297107, 681227951,
            10108438218, 10108674170, 13476558714, 8856333029, 13476558710,
            9191074329, 10108438224, 7675966173, 312297106, 3393291362,
            ]}
]

yellowJeepSouth = [
        {"bound": "south", "nodeList": [
            1872304943, 9191109020, 3393291352, 7676166990, 1872304937,
            9191121906, 529853828, 5549726320, 7675801284, 8318725917,
            854202919, 7675801282, 6257635296, 2627302000, 10979944928,
            7676009004, 13476558708, 2627302003, 2627302029, 854186888,
            9191058308, 12206389590, 2892712059, 12206389589, 2627301958,
            9191058305, 1841740882, 13503867537, 7675766470, 846538010, 
            1841758381, 7675766449, 1841757566, 1841746736, 5367974291,
            5367974293, 7675071639, 313913653, 9193923314, 1872304768,
            9193923301, 313918603, 9193923289, 3339587462, 3339627292, 3339627291,
            7205622651, 457276605, 7676073006, 10879937416
            ]}
]

sandJeep = [
    {"bound": "south", "nodeList": [
        9524704887, 6405400943, 518296277, 7675675876, 9210461745, 9210461750, 10113363114, 518311099, 7675833312, 7675833337,
        7675833335, 485943267, 485943229, 485943184, 7675833333, 8333738175, 3341120580, 4712003759, 8675274097, 1844997074,
        2674509944, 518303252, 8277171944, 13477093306, 13477093315, 8277171943, 518306880, 1540323434, 9210326682, 6400092070,
        9210326680, 8277171946, 8277171945, 8277171948, 2565897721, 678603284, 369890130, 369890164, 369889822, 369889673,
        369889690, 369889779, 336120969, 336121084, 9196185211, 9196185210, 259451787, 9196185178, 9196185200, 9196185216, 678598607,
        336118916, 336118905, 518325254, 5237457718, 7198611444, 1553283109, 12926128516, 259451803, 1553282070, 8856675041,
        6266023669, 259451806, 259451808, 5381130548, 8678858237, 9208235025, 825688443, 538130945, 8678858244, 8678819203
    ]},
    {"bound": "north", "nodeList": [
        8678819191, 8678819203, 8678858244, 538130945, 825688443, 9208235025, 8678858237, 5381130548, 259451808, 259451806,
        6266023669, 8856675041, 1553282070, 259451803, 12926128516, 1553283109, 259451796, 5237457718, 518325254, 336118905,
        336118916, 678598607, 9196185216, 9196185200, 9196185178, 259451787, 9196185210, 9196185211, 336121084, 336120969, 369889779,
        369889690, 369889673, 369889822, 369890164, 369890130, 678603284, 2565897721, 8277171948, 8277171945, 8277171946, 9210326680,
        6400092070, 9210326680, 6400092070, 9210326682, 1540323434, 518306880, 8277171943, 13477093315, 13477093306, 8277171944, 518303252,
        2674509944, 1844997074, 8675274097, 4712003759, 3341120580, 8333738175, 7675833333, 485943184, 485943229, 485943267,
        7675833335, 7675833337, 7675833312, 518311099, 10113363114, 9210461750, 9210461745, 7675675876, 9524704858, 9524704869
    ]
     }
]

whiteJeep = [
    {"bound": "south", "nodeList": [
        9524704858, 7675675876, 518296277, 314843398, 1844973495, 314843401, 1844973218, 314843403, 518296278, 518296279,
        1844973124, 7675656522, 7675742390, 518309049, 279718170, 12772735473, 10112895406, 10113692705, 9925714422, 660220671,
        10110861641, 10110861649, 279718165, 336130781, 538132313, 538132312, 10110693161, 660220595, 279718162, 12850537637,
        8675029605, 507044666, 6266016976, 10097950830, 279718161, 10100802914, 8679215108, 10100802912, 4952801802, 1853859097,
        7675715676, 7675715675, 672571151, 679318602, 672571146, 10100601560, 679314330, 279718156, 279718155, 529856382,
        4952801798, 457281472, 517417851, 279718148, 3556332238, 6831097023, 3393291352, 7676166990, 1872304937, 9191121906
    ]},
    {"bound": "north", "nodeList": [
        529853828, 5549726320, 312296969, 7675835589, 3393291362, 1872304943, 9191109020, 3393291365, 6831097024, 3556332238, 279718148, 517417851,
        457281472, 4952801798, 529856382, 279718155, 279718156, 679314330, 10100601560, 672571146, 679318602, 672571151,
        7675715675, 7675715676, 1853859097, 4952801802, 10100802912, 8679215108, 10100802914, 279718161, 10097950830, 6266016976,
        507044666, 8675029605, 12850537637, 279718162, 660220595, 10110693161, 538132312, 538132313, 336130781, 279718165,
        10110861649, 10110861641, 660220671, 9925714422, 10113692705, 10112895406, 12772735473, 279718170, 518309049,
        7675742390, 7675656522, 1844973124, 518296279, 518296278, 314843403, 1844973218, 314843401, 1844973495, 314843398,
        518296277, 7675675876, 9524704858
    ]}
]

blueJeep = [
    {"bound": "south", "nodeList": [
        6514717897, 2186720970, 6514889307, 2186720969, 2186718337, 3341100960, 3341100958, 3388973405, 3388973406, 3388973407, 3341100892, 317176865, 8992824705, 457888678, 9196110871, 7675063407, 6846711041, 9196090037, 12035336006, 12035336005, 449659809, 9196090029, 12595304824, 449659810, 11258891594, 11258891598, 13692022598, 9196021615, 527827933, 10101099008, 9196021616, 7675100203, 7675695295, 527827924, 854339638, 13479138054, 13479138061, 7675063465, 7844953666, 10095114316, 9196021556, 854338664, 449658386, 9196021573, 9196021557, 6118354752, 449658387, 311718967, 9196000168, 7676313057, 9196000167, 313908262, 7676077658, 3343727920, 3343727918, 3343727917, 11269451251, 313908392, 313910357, 438033707, 7713520922, 7693476941, 313909113, 438033711, 457306235, 3339627301, 7675735513, 3339627332, 3339627335, 3339627323, 9183498724, 3339627340, 312291686, 7675733173, 8856524323, 518261998, 13511942695, 6828588389, 10113743412, 313910490, 312291693, 311674987, 13509227789, 9183623311, 13509227795, 13509227786, 8302829218, 13509227798, 9183623297, 6355364996, 6870297706, 13509227792, 312294586, 6828588336, 9191021573, 9191021570, 312297509
    ]},
    {"bound": "north", "nodeList": [
        312297107, 681227951, 10108438218, 10108674170, 13476558714, 8856333029, 13476558710, 9191074329, 10108438224, 7675966173, 312297106, 1872304939, 7675835597, 7675835590, 312296107, 312296082, 514546861, 514546874, 8304926821, 514546883, 514546860, 514546889, 681226526, 8323299017, 312294586, 13509227792, 6870297706, 6355364996, 9183623297, 13509227798, 8302829218, 13509227786, 13509227795, 9183623311, 13509227789, 311674987, 312291693, 313910490, 10113743412, 6828588389, 13511942695, 518261998, 8856524323, 7675733173, 312291686,
        3339627326, 3339627327, 3339627338, 3378951301, 11084024153, 3339627301, 457306235, 438033711, 313909113, 7693476941, 7713520922, 438033707, 313910357, 313908392, 11269451251, 3343727917, 3343727918, 3343727920, 7676077658, 313908262, 9196000167, 7676313057, 9196000168, 311718967, 449658387, 6118354752, 9196021557, 9196021573, 449658386, 854338664, 9196021556, 10095114316, 7844953666, 7675063465, 13479138061, 13479138054, 854339638, 527827924, 7675695295, 7675100203, 9196021616, 10101099008, 527827933, 9196021615, 13692022598, 11258891598, 11258891594, 449659810, 12595304824, 9196090029, 449659809, 12035336005, 12035336006, 9196090037, 6846711041, 7675063407, 9196110871, 457888678, 8992824705, 317176865,
        7686348577, 3388973409, 6514686662, 6514686664, 3341100879, 3341100875,
        3388973403,
        3341100919,
        3341100936,
        3341100938,
        6514686665,
        2322891845,
        2186718351,
        2186718334
    ]}
]

orangeJeep = [
    {"bound": "north", "nodeList": [
  7676065558, 7676065559, 5549727821, 457281145, 457281146, 1872304904, 311685890, 7198611405, 529853829, 7676197626,
  9293818839, 311297632, 9293818837, 9293818827, 7676065530, 1606318342, 529853880, 8323228925, 529860639, 901134849,
  529853883, 529853872, 529853873, 7510686963, 7510028781, 12206477907, 12206477905, 7678730479, 12206477911, 9293908701,
  529853884, 7694154853, 7676633045, 7676661774, 1851459260, 7676661764, 8856235108, 12037929655, 9208032533, 9293908661,
  9243306534, 13179728034, 13179728032, 9293919625, 12016011229, 12016011235, 259451830, 12037929652, 529860604, 336122948,
  8856189205, 336121970, 9209281670, 8277217104, 8278463175, 8278463180, 8679312913, 336122182, 259451826, 9209305837,
  8679100797, 8679100789, 8679100788, 336276012, 8679523882, 259451824, 13476211786, 336283655
]},
    {"bound": "south", "nodeList": [
  679182315, 336283655, 13476211786, 259451824, 8679523882, 336276012, 8679100788, 8679100789, 8679100797, 9209305837,
  259451826, 336122182, 8679312913, 8278463180, 8278463175, 8277217104, 9209281670, 336121970, 8856189205, 336122948,
  529860604, 12037929652, 259451830, 12016011227, 13179728031, 529860601, 7676661778, 7676711675, 7676661777, 8856235114,
  12037929667, 259451831, 9208032533, 12037929655, 8856235108, 7676661764, 1851459260, 7676661774, 7676633045, 7694154853,
  529853884, 9293908701, 12206477911, 7678730479, 12206477905, 12206477907, 7510028781, 7510686963, 529853873, 529853872,
  529853883, 901134849, 529860639, 8323228925, 529853880, 1606318342, 7676065530, 9293818827, 9293818837, 311297632,
  9293818839, 7676197626, 529853829, 7198611405, 311685890, 1872304904, 457281146, 457281145, 5549727821, 7676065559,
  7676065558
]}
]

daucpointSouth = [
    {"bound": "south", "nodeList": [
    9187628054, 8278178678, 13250223434, 7674980613, 10111017985, 9187628055, 7357508289, 372577036, 522301886, 372578357,
    7357508288, 372546258, 538858115, 522298465, 7675045550, 522298220, 10911031191, 522298412, 11084719234, 11084719235,
    283119886, 6266037530, 372443621, 360697784, 372528399, 6732277546, 11070696329, 9187839297, 6732277551, 6732277555,
    6732277553, 11138324288, 6732277533, 11138333012, 6732277535, 11138333010, 7673464378, 8682825680, 2748923904, 8682825683,
    8682825685, 13504279964, 7357315008, 13504279973, 7357315004, 7357315018, 7357332269, 315735804, 508035441, 314854754,
    7695561764, 518296269, 6814692361, 3289714300, 3289714306, 1540323475, 3289714286, 3289714296, 1540323472, 3289717496,
    1540323469, 5548940641, 6814957201, 507044703, 8333736630, 1540323460, 10897890037, 1540323459, 10981724689, 1540323467,
    8693517510, 10298120829, 7920872738, 10298112466
]}
]

daucpointNorth = [
    {"bound": "north", "nodeList": [
    10298120821, 10298112466, 7920872738, 8693517510, 1540323467,
    10981724689, 1540323459, 10897890037, 1540323460, 8333736630,
    507044703, 6814957201, 5548940641, 1540323469, 3289717496,
    1540323472, 3289714296, 3289714287, 3289714288, 3289714289,
    5548939409, 5548939411, 3289714301, 3289714294, 11468813213,
    518296269, 7695561764, 314854754, 508035441, 315735804,
    7357332269, 7357315018, 7357315004, 13504279973, 7357315008,
    13504279964, 8682825685, 8682825683, 2748923904, 8682825681,
    1842375233, 6732277534, 8682963010, 7675047513, 6727297893,
    11138324289, 9188048889, 9188048896, 315737081, 8682986717,
    8682963011, 6732280689, 1703686178, 12093573, 6727297891,
    6732133037, 9187839296, 6732277546, 372528399, 360697784,
    372443621, 6266037530, 283119886, 7676449887, 7678608552,
    10911031181, 1355872786, 1355872754, 1355950582,
]}
]

addJeepRoute(redJeepNorth, "SapangBato-Angeles-Northbound")
addJeepRoute(redJeepsouth, "SapangBato-Angeles-Southbound")
addJeepRoute(purpleJeepList, "Checkpoint-HolyAngelUniversity-Highway-Angeles")
addJeepRoute(MabalacatDauAngelesSouth, "Mabalacat-Dau-Angeles-Southbound")
addJeepRoute(MabalacatDauAngelesNorth, "Mabalacat-Dau-Angeles-Northbound")
addJeepRoute(greyJeep, 'Checkpoint-HolyAngelUniversity')
addJeepRoute(greepJeep, 'Marisol-Pampang')
addJeepRoute(yellowJeepNorth, 'Villa-Pampang-Northbound')
addJeepRoute(yellowJeepSouth, 'Villa-Pampang-Southbound')
addJeepRoute(sandJeep, 'Maingate-Friendship')
addJeepRoute(whiteJeep, 'Checkpoint-Hensonville-Holy')
addJeepRoute(blueJeep, 'Angeles-Pulung-Marugul-via-Pandan')
addJeepRoute(orangeJeep, 'Sunset-Nepo-Angeles')
addJeepRoute(daucpointSouth, 'Clark-Main Gate-Dau-Southbound')
addJeepRoute(daucpointNorth, 'Clark-Main Gate-Dau-Northbound')  

KeyboardInterrupt: 

In [21]:
# Add individual jeep

import copy
import json
import os

nodeIdIndex = 0
edgeIdIndex = 0

def addJeepRoute(
        nodeList: dict[str, list[int]], 
        routeName: str
    ):
    '''
    Creates a node and edges file in GeoJSON format from a list of nodes that constitutes a path aligned to the jeepney route

    Throws an error if an osmid from the node list does not exist in the graph
    Throws an error if two adjacent nodes in the list do not have an edge connecting one another in the graph

    
    Parameters
    ----------
    nodeList
        It should be a list of nodes that constitutes a path aligned to the jeepney route.
        It will throw an error if the node_id in the graph does not exist, or if two adjacent
        nodes in the list dont have an edge connecting each other

    routeName
        Name of the jeepney route, preferably what the label on the side of the jeepney says

    Returns
    ----------
    None
    '''
    global nodeIdIndex
    global edgeIdIndex

    def max_osmid(base_path, suffix):
        max_id = 0
        for route in os.listdir(base_path):
            for file in os.listdir(f"{base_path}/{route}"):
                if file.endswith(suffix):
                    with open(f"{base_path}/{route}/{file}") as f:
                        data = json.load(f)
                    ids = [int(feat['properties']['osmid']) for feat in data['features']]
                    if ids:
                        max_id = max(max_id, max(ids))
        return max_id

    edgeIdIndex = max(
        max_osmid("graph/transit_graph/trike_routes", "edges.geojson"),
        max_osmid("graph/transit_graph/jeep_routes", "edges.geojson")
    ) + 1

    nodeIdIndex = max(
        max_osmid("graph/transit_graph/trike_routes", "nodes.geojson"),
        max_osmid("graph/transit_graph/jeep_routes", "nodes.geojson")
    ) + 1

    
    osmIdDict = {}

    # Tracks the ids of walking nodes' equivalent mode nodes


    # Reads the nodes, edges json of the original graph
    with open('graph/original_graph/mabalacatAngeles_nodes.geojson', 'r') as file:
        nodes_jsonData = json.load(file)
    with open('graph/original_graph/mabalacatAngeles_edges.geojson', 'r') as file:
        edges_jsonData = json.load(file)

    # Json file of the jeepney route nodes to write
    jeepRouteNodesJson = {
        "type": "FeatureCollection",
        "name": f"{routeName}_nodes",
        "crs": {
            "type": "name", 
            "properties": { 
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84" 
            } 
        },
        "features": [

        ]
    }

    # Json file of the jeepney route edges to write
    jeepRouteEdgesJson = {
        "type": "FeatureCollection",
        "name": f"{routeName}_edges",
        "crs": {
            "type": "name", 
            "properties": { 
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84" 
            } 
        },
        "features": [

        ]
    }
    
    nodesVisited = {}

    fullList = [
        {"bound": item["bound"], "id": node}
        for item in nodeList
        for node in item["nodeList"]
    ]

    # --------------------- FIRST NODE -------------------------#
    # Looks for the first node id from the route list 
    # inside the graph and gets data about its original node

    jeepNodeToAdd = None
    for i in nodes_jsonData['features']:
        if i['properties']['osmid'] == fullList[0]['id']:
            jeepNodeToAdd = copy.deepcopy(i)

            # Creates a unique osm_id for the node of the route
            # by this formula (105002 * node_id + 67203817)
            # newOsmId = createNewOsmId(nodeList[0])

            # Assigns id to new mode node
            newOsmId = nodeIdIndex
            osmIdDict[fullList[0]['id']] = newOsmId

            # Changes old osm_id to new one
            jeepNodeToAdd['properties']['osmid'] = newOsmId

            jeepNodeToAdd['properties']['mode'] = {
                "type": "jeep",
                "details": {
                    "name": routeName,
                    "bound": fullList[0]['bound']
                }
            }

            # Makes sure that newly added node is not duplicated in
            # case it is traversed again in the route
            nodesVisited[fullList[0]['id']] = copy.deepcopy(jeepNodeToAdd)
            break

    if (jeepNodeToAdd):
        jeepRouteNodesJson["features"].append(jeepNodeToAdd)
    else:
        raise KeyError(f"Node with id \"{fullList[0]['id']}\" is not in the graph.")

    # Outbound and inbound jeep node weights
    # 470 for inbound weight is designated to add penalty for boarding,
    # used to discourage the algorithm from picking short jeepney trips
    outJeepWeight = 10; inJeepWeight = 300

    # Function that creates a new feature for the edgesJson['features'] dictionary
    inOut_edgeFeature = lambda u, v, id, nodeFeatures, direction: {
        "type": "Feature",
        "properties": {
            "u": u,
            "v": v,
            "key": 0,
            "osmid": str(id),
            "length": 0.01,
            "weight": inJeepWeight if direction=='in' else outJeepWeight if direction=='out' else '0'
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']],
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']]
            ]
        }
    }

    # Add inbound jeep edge
    jeepRouteEdgesJson['features'].append(inOut_edgeFeature(fullList[0]['id'], newOsmId, edgeIdIndex, jeepNodeToAdd, 'in'))

    # Add outbound jeep edge
    jeepRouteEdgesJson['features'].append(inOut_edgeFeature(newOsmId, fullList[0]['id'], edgeIdIndex, jeepNodeToAdd, 'out'))

    # ---------------- SECOND TO THE LAST NODE -------------------#
    for m in range(1, len(fullList)):
        nodeIdIndex += 1; edgeIdIndex += 1
        jeepNodeToAdd = None

        # Check if node has not been visited already in the route
        if fullList[m]['id'] not in nodesVisited.keys():

            # Find node in the graph
            for i in nodes_jsonData['features']:

                # Check if node matches
                if i['properties']['osmid'] == fullList[m]['id']:

                    # Get node feature details
                    jeepNodeToAdd = copy.deepcopy(i)

                    newOsmId = nodeIdIndex
                    osmIdDict[fullList[m]['id']] = newOsmId

                    jeepNodeToAdd['properties']['osmid'] = newOsmId

                    # Add walking node and its features to visited list
                    nodesVisited[fullList[m]['id']] = copy.deepcopy(jeepNodeToAdd)
                    break

            if (jeepNodeToAdd):
                jeepNodeToAdd["properties"]["mode"] = {
                    "type": "jeep",
                    "details": {
                        "name": routeName,
                        "bound": fullList[m]['bound']
                    }
                }
                jeepRouteNodesJson["features"].append(jeepNodeToAdd)
            else:
                raise KeyError(f"Node with id \"{fullList[m]['id']}\" is not in the graph.")

            # Inbound jeep edge
            jeepRouteEdgesJson['features'].append(inOut_edgeFeature(fullList[m]['id'], newOsmId, edgeIdIndex, jeepNodeToAdd, 'in'))

            # Outbound jeep edge
            jeepRouteEdgesJson['features'].append(inOut_edgeFeature(newOsmId, fullList[m]['id'], edgeIdIndex, jeepNodeToAdd, 'out'))

        else:
            newOsmId = osmIdDict[fullList[m]['id']]

        edgeDetails = None

        for i in edges_jsonData['features']:
            if i['properties']['u'] == fullList[m-1]['id'] and i['properties']['v'] == fullList[m]['id'] and i['properties']['key'] == 0:
                edgeDetails = copy.deepcopy(i)
                break
        
        jeepSpeed = 3.89 # m/s
        if (edgeDetails):
            geometry = edgeDetails['geometry']
            length = edgeDetails['properties']['length']
            edgeIdIndex += 1

            # Calculated from length by speed which corresponds to how long
            # it takes to traverse an edge based on jeepney's average speed
            # in seconds
            weight = round(round(length, 2) / jeepSpeed, 2)

            jeepRouteEdgesJson['features'].append({
                "type": "Feature",
                "properties": {
                    "u": osmIdDict[fullList[m-1]['id']],
                    "v": newOsmId,
                    "key": 0,
                    "osmid": str(edgeIdIndex),
                    "length": length,
                    "weight": weight
                },
                "geometry": geometry
            })
        else:
            warningInput = input(
                f"Edge with u({fullList[m-1]['id']}) and v({fullList[m]['id']}) pair does not exist!\n" \
                "Are you sure you want to continue? Route may have disconnected paths.\n" \
                "Input \"yes\" to continue with exporting: "
            )

            if warningInput.lower() != 'yes':
                raise KeyError('Edge with corresponding (u,v) pair does not exist in the graph.')
            
    if not os.path.exists(f"graph/transit_graph/jeep_routes/{routeName}"):
        os.makedirs(f"graph/transit_graph/jeep_routes/{routeName}")

    with open(f'graph/transit_graph/jeep_routes/{routeName}/{routeName}_nodes.geojson', 'w') as file:
        json.dump(jeepRouteNodesJson, file, indent=1)

    with open(f'graph/transit_graph/jeep_routes/{routeName}/{routeName}_edges.geojson', 'w') as file:
        json.dump(jeepRouteEdgesJson, file, indent=1)

# blueJeep = [
#     {"bound": "south", "nodeList": [
#         6514717897, 2186720970, 6514889307, 2186720969, 2186718337, 3341100960, 3341100958, 3388973405, 3388973406, 3388973407, 3341100892, 317176865, 8992824705, 457888678, 9196110871, 7675063407, 6846711041, 9196090037, 12035336006, 12035336005, 449659809, 9196090029, 12595304824, 449659810, 11258891594, 11258891598, 13692022598, 9196021615, 527827933, 10101099008, 9196021616, 7675100203, 7675695295, 527827924, 854339638, 13479138054, 13479138061, 7675063465, 7844953666, 10095114316, 9196021556, 854338664, 449658386, 9196021573, 9196021557, 6118354752, 449658387, 311718967, 9196000168, 7676313057, 9196000167, 313908262, 7676077658, 3343727920, 3343727918, 3343727917, 11269451251, 313908392, 313910357, 438033707, 7713520922, 7693476941, 313909113, 438033711, 457306235, 3339627301, 7675735513, 3339627332, 3339627335, 3339627323, 9183498724, 3339627340, 312291686, 7675733173, 8856524323, 518261998, 13511942695, 6828588389, 10113743412, 313910490, 312291693, 311674987, 13509227789, 9183623311, 13509227795, 13509227786, 8302829218, 13509227798, 9183623297, 6355364996, 6870297706, 13509227792, 312294586, 6828588336, 9191021573, 9191021570, 312297509
#     ]},
#     {"bound": "north", "nodeList": [
#         312297107, 681227951, 10108438218, 10108674170, 13476558714, 8856333029, 13476558710, 9191074329, 10108438224, 7675966173, 312297106, 1872304939, 7675835597, 7675835590, 312296107, 312296082, 514546861, 514546874, 8304926821, 514546883, 514546860, 514546889, 681226526, 8323299017, 312294586, 13509227792, 6870297706, 6355364996, 9183623297, 13509227798, 8302829218, 13509227786, 13509227795, 9183623311, 13509227789, 311674987, 312291693, 313910490, 10113743412, 6828588389, 13511942695, 518261998, 8856524323, 7675733173, 312291686,
#         3339627326, 3339627327, 3339627338, 3378951301, 11084024153, 3339627301, 457306235, 438033711, 313909113, 7693476941, 7713520922, 438033707, 313910357, 313908392, 11269451251, 3343727917, 3343727918, 3343727920, 7676077658, 313908262, 9196000167, 7676313057, 9196000168, 311718967, 449658387, 6118354752, 9196021557, 9196021573, 449658386, 854338664, 9196021556, 10095114316, 7844953666, 7675063465, 13479138061, 13479138054, 854339638, 527827924, 7675695295, 7675100203, 9196021616, 10101099008, 527827933, 9196021615, 13692022598, 11258891598, 11258891594, 449659810, 12595304824, 9196090029, 449659809, 12035336005, 12035336006, 9196090037, 6846711041, 7675063407, 9196110871, 457888678, 8992824705, 317176865,
#         7686348577, 3388973409, 6514686662, 6514686664, 3341100879, 3341100875,
#         3388973403,
#         3341100919,
#         3341100936,
#         3341100938,
#         6514686665,
#         2322891845,
#         2186718351,
#         2186718334
#     ]}
# ]

orangeJeep = [
    {"bound": "north", "nodeList": [
  7676065558, 7676065559, 5549727821, 457281145, 457281146, 1872304904, 311685890, 7198611405, 529853829, 7676197626,
  9293818839, 311297632, 9293818837, 9293818827, 7676065530, 1606318342, 529853880, 8323228925, 529860639, 901134849,
  529853883, 529853872, 529853873, 7510686963, 7510028781, 12206477907, 12206477905, 7678730479, 12206477911, 9293908701,
  529853884, 7694154853, 7676633045, 7676661774, 1851459260, 7676661764, 8856235108, 12037929655, 9208032533, 9293908661,
  9243306534, 13179728034, 13179728032, 9293919625, 12016011229, 12016011235, 259451830, 12037929652, 529860604, 336122948,
  8856189205, 336121970, 9209281670, 8277217104, 8278463175, 8278463180, 8679312913, 336122182, 259451826, 9209305837,
  8679100797, 8679100789, 8679100788, 336276012, 8679523882, 259451824, 13476211786, 336283655
]},
    {"bound": "south", "nodeList": [
  679182315, 336283655, 13476211786, 259451824, 8679523882, 336276012, 8679100788, 8679100789, 8679100797, 9209305837,
  259451826, 336122182, 8679312913, 8278463180, 8278463175, 8277217104, 9209281670, 336121970, 8856189205, 336122948,
  529860604, 12037929652, 259451830, 12016011227, 13179728031, 529860601, 7676661778, 7676711675, 7676661777, 8856235114,
  12037929667, 259451831, 9208032533, 12037929655, 8856235108, 7676661764, 1851459260, 7676661774, 7676633045, 7694154853,
  529853884, 9293908701, 12206477911, 7678730479, 12206477905, 12206477907, 7510028781, 7510686963, 529853873, 529853872,
  529853883, 901134849, 529860639, 8323228925, 529853880, 1606318342, 7676065530, 9293818827, 9293818837, 311297632,
  9293818839, 7676197626, 529853829, 7198611405, 311685890, 1872304904, 457281146, 457281145, 5549727821, 7676065559,
  7676065558
]}
]

daucpointSouth = [
    {"bound": "south", "nodeList": [
    9187628054, 8278178678, 13250223434, 7674980613, 10111017985, 9187628055, 7357508289, 372577036, 522301886, 372578357,
    7357508288, 372546258, 538858115, 522298465, 7675045550, 522298220, 10911031191, 522298412, 11084719234, 11084719235,
    283119886, 6266037530, 372443621, 360697784, 372528399, 6732277546, 11070696329, 9187839297, 6732277551, 6732277555,
    6732277553, 11138324288, 6732277533, 11138333012, 6732277535, 11138333010, 7673464378, 8682825680, 2748923904, 8682825683,
    8682825685, 13504279964, 7357315008, 13504279973, 7357315004, 7357315018, 7357332269, 315735804, 508035441, 314854754,
    7695561764, 518296269, 6814692361, 3289714300, 3289714306, 1540323475, 3289714286, 3289714296, 1540323472, 3289717496,
    1540323469, 5548940641, 6814957201, 507044703, 8333736630, 1540323460, 10897890037, 1540323459, 10981724689, 1540323467,
    8693517510, 10298120829, 7920872738, 10298112466
]}
]

daucpointNorth = [
    {"bound": "north", "nodeList": [
    10298120821, 10298112466, 7920872738, 8693517510, 1540323467,
    10981724689, 1540323459, 10897890037, 1540323460, 8333736630,
    507044703, 6814957201, 5548940641, 1540323469, 3289717496,
    1540323472, 3289714296, 3289714287, 3289714288, 3289714289,
    5548939409, 5548939411, 3289714301, 3289714294, 11468813213,
    518296269, 7695561764, 314854754, 508035441, 315735804,
    7357332269, 7357315018, 7357315004, 13504279973, 7357315008,
    13504279964, 8682825685, 8682825683, 2748923904, 8682825681,
    1842375233, 6732277534, 8682963010, 7675047513, 6727297893,
    11138324289, 9188048889, 9188048896, 315737081, 8682986717,
    8682963011, 6732280689, 1703686178, 12093573, 6727297891,
    6732133037, 9187839296, 6732277546, 372528399, 360697784,
    372443621, 6266037530, 283119886, 7676449887, 7678608552,
    10911031181, 1355872786, 1355872754, 1355950582,
]}
]

# addJeepRoute(blueJeep, 'Angeles-Pulung-Marugul-via-Pandan')

# addJeepRoute(orangeJeep, 'Sunset-Nepo-Angeles')

# addJeepRoute(daucpointSouth, 'Clark-Main Gate-Dau-Southbound')
addJeepRoute(daucpointNorth, 'Clark-Main Gate-Dau-Northbound')


In [76]:
import csv, os, json

def addTrikeRoute(
        terminalList: list[int],
        subgraphDir: str,
        todaName: str
    ):
    '''
    Creates a node and edges file in GeoJSON format from a list of nodes that constitutes
    a subgraph aligned to the tricycle TODA's coverage area

    Throws an error if an osmid from nodeList does not exist in G
    Throws an error if an osmid from terminalList is not present in nodeList

    Parameters
    ----------
    terminalList
        List of osmids (must also be present in nodeList) that act as the gateway/boarding
        points between the walking graph and the tricycle subgraph

    nodeList
        List of every osmid from G to be included as part of the tricycle subgraph

    todaName
        Name of the TODA (Tricycle Operators and Drivers' Association), preferably the
        terminal's official name

    Returns
    ----------
    None
    '''

    osmIdDict = {}

    nodeList = []

    
    with open(subgraphDir, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        next(reader)  # Skip header

        for row in reader:
            if row[1] != 'osmid' and int(row[1]) not in nodeList:
                nodeList.append(int(row[1]))  # osmid column


    def max_osmid(base_path, suffix):
        max_id = 0
        for route in os.listdir(base_path):
            for file in os.listdir(f"{base_path}/{route}"):
                if file.endswith(suffix):
                    with open(f"{base_path}/{route}/{file}") as f:
                        data = json.load(f)
                    ids = [int(feat['properties']['osmid']) for feat in data['features']]
                    if ids:
                        max_id = max(max_id, max(ids))
        return max_id

    edgeIdIndex = max(
        max_osmid("graph/transit_graph/trike_routes", "edges.geojson"),
        max_osmid("graph/transit_graph/jeep_routes", "edges.geojson")
    )

    nodeIdIndex = max(
        max_osmid("graph/transit_graph/trike_routes", "nodes.geojson"),
        max_osmid("graph/transit_graph/jeep_routes", "nodes.geojson")
    )


    # Json file of the tricycle subgraph nodes to write
    trikeNodesJson = {
        "type": "FeatureCollection",
        "name": f"{todaName}_nodes",
        "crs": {
            "type": "name",
            "properties": {
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
            }
        },
        "features": [

        ]
    }

    # Json file of the tricycle subgraph edges to write
    trikeEdgesJson = {
        "type": "FeatureCollection",
        "name": f"{todaName}_edges",
        "crs": {
            "type": "name",
            "properties": {
                "name": "urn:ogc:def:crs:OGC:1.3:CRS84"
            }
        },
        "features": [

        ]
    }

    # Builds a GeoJSON Point feature for an osmid, pulling attributes directly from G
    def nodeFeatureFromG(osmid):
        nodeData = G.nodes[osmid]
        return {
            "type": "Feature",
            "properties": {
                "osmid": osmid,
                "y": nodeData.get('y'),
                "x": nodeData.get('x'),
                "street_count": int(nodeData.get('street_count')),
                "highway": nodeData.get('highway'),
                "ref": nodeData.get('ref')
            },
            "geometry": {
                "type": "Point",
                "coordinates": [nodeData.get('x'), nodeData.get('y')]
            }
        }

    # Builds a LineString geometry for an edge, using G's edge geometry if present,
    # otherwise falling back to a straight line between the two node coordinates
    def edgeGeometryFromG(u, v, edgeData):
        if 'geometry' in edgeData:

            coordList = []
            for coord in edgeData['geometry'].coords:
                coordList.append(tuple(coord))

            return {
                "type": "LineString",
                "coordinates": coordList
            }

        return {
            "type": "LineString",
            "coordinates": [
                [G.nodes[u]['x'], G.nodes[u]['y']],
                [G.nodes[v]['x'], G.nodes[v]['y']]
            ]
        }

    outTrikeWeight = 10; inTrikeWeight = 300

    inOut_edgeFeature = lambda u, v, id, nodeFeatures, direction: {
        "type": "Feature",
        "properties": {
            "u": u,
            "v": v,
            "key": 0,
            "osmid": str(id),
            "length": 0.01,
            "weight": inTrikeWeight if direction == 'in' else outTrikeWeight if direction == 'out' else '0'
        },
        "geometry": {
            "type": "LineString",
            "coordinates": [
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']],
                [nodeFeatures['properties']['x'], nodeFeatures['properties']['y']]
            ]
        }
    }

    # -------------------- CREATE TRIKE NODES -------------------- #
    # For every node_id in nodeList, get its data from G
    # and add it to trikeNodesJson as a new trike-mode node

    for ogId in nodeList:
        if ogId not in walkingG.nodes:
            raise KeyError(f"Node with id \"{ogId}\" is not in G.")

        trikeNodeToAdd = nodeFeatureFromG(ogId)

        nodeIdIndex += 1
        newOsmId = nodeIdIndex
        osmIdDict[ogId] = newOsmId

        trikeNodeToAdd['properties']['osmid'] = newOsmId
        trikeNodeToAdd['properties']['mode'] = {
            "type": "trike",
            "details": {
                "name": todaName,
                "terminal": True if ogId in terminalList else False
            }
        }

        trikeNodesJson["features"].append(trikeNodeToAdd)

        # Outbound trike edge (trike node -> walking node)
        edgeIdIndex += 1
        trikeEdgesJson['features'].append(
            inOut_edgeFeature(newOsmId, ogId, edgeIdIndex, trikeNodeToAdd, 'out')
        )

    # Makes sure every terminal osmid is actually part of nodeList
    for terminalId in terminalList:
        if terminalId not in osmIdDict:
            raise KeyError(f"Terminal node with id \"{terminalId}\" is not in nodeList.")

    # -------------------- CREATE SUBGRAPH EDGES -------------------- #
    # Only create an edge between two newly created trike nodes if their
    # corresponding original nodes (u, v) both exist in nodeList AND
    # already have an edge connecting them in G. All u-v pairs are kept
    # as-is, direction preserved.

    trikeSpeed = 6.48  # m/s

    nodeListSet = set(nodeList)

    for u, v, key, data in G.edges(keys=True, data=True):
        if key != 0:
            continue

        if u in nodeListSet and v in nodeListSet:
            edgeIdIndex += 1

            length = data.get('length')
            weight = round(round(length, 2) / trikeSpeed, 2)

            trikeEdgesJson['features'].append({
                "type": "Feature",
                "properties": {
                    "u": osmIdDict[u],
                    "v": osmIdDict[v],
                    "key": 0,
                    "osmid": str(edgeIdIndex),
                    "length": length,
                    "weight": weight
                },
                "geometry": edgeGeometryFromG(u, v, data)
            })

    # -------------------- CREATE TERMINAL BOARDING EDGES -------------------- #
    # Only nodes in terminalList get inbound/outbound edges connecting them
    # to their corresponding original (walking) node. Non-terminal trike
    # nodes never connect back to the walking graph.



    for terminalId in terminalList:
        newOsmId = osmIdDict[terminalId]

        terminalNodeFeature = None
        for i in trikeNodesJson['features']:
            if i['properties']['osmid'] == newOsmId:
                terminalNodeFeature = i
                break

        # Inbound trike edge (walking node -> trike terminal node)
        edgeIdIndex += 1
        trikeEdgesJson['features'].append(
            inOut_edgeFeature(terminalId, newOsmId, edgeIdIndex, terminalNodeFeature, 'in')
        )



    # -------------------- WRITE OUTPUT -------------------- #
    if not os.path.exists(f"graph/transit_graph/trike_routes/{todaName}"):
        os.makedirs(f"graph/transit_graph/trike_routes/{todaName}")

    with open(f'graph/transit_graph/trike_routes/{todaName}/{todaName}_nodes.geojson', 'w') as file:
        json.dump(trikeNodesJson, file, indent=1)

    with open(f'graph/transit_graph/trike_routes/{todaName}/{todaName}_edges.geojson', 'w') as file:
        json.dump(trikeEdgesJson, file, indent=1)

# addTrikeRoute([259451803], r"graph\test_graph\todatxt\atdaanunas.txt", 'ATDA TODA')
# addTrikeRoute([518296271], r"graph\test_graph\todatxt\mvfstoda.txt", 'MVFS TODA')

# addTrikeRoute([313908392, 5242235029], r"graph\test_graph\todatxt\miningtoda.txt", 'MINING TODA')

# Official
addTrikeRoute([456858251, 458615236, 7678722356, 373031689, 7673405093], r"graph\test_graph\todatxt\srmm.txt", 'SRMMMMRCDM TODA')
addTrikeRoute([518296272, 8679646653, 7357917580, 1843464643, 7679097533, 457888892], r"graph\test_graph\todatxt\zone1.txt", 'Zone 1')






In [8]:
import os, json

a = []
for i in os.listdir(r"graph\transit_graph\trike_routes"):
    with open(rf"graph\transit_graph\trike_routes\{i}\{i}_edges.geojson") as f:

        edgeJson = json.load(f)

        for j in edgeJson['features']:
            if j['properties']['u'] > 13000 and j['properties']['u'] not in a:
                print(f"{i} terminal node: {j['properties']['u']}")
                a.append(j['properties']['u'])
            elif j['properties']['v'] > 13000 and j['properties']['v'] not in a:
                print(f"{i} terminal node: {j['properties']['v']}")
                a.append(j['properties']['v'])

ATDA TODA terminal node: 259451803
CUAYAN TODA terminal node: 1235692172
DAMA DECA TODA terminal node: 2995803925
MCMCHSD TODA terminal node: 6406813983
MINING TODA terminal node: 313908392
MINING TODA terminal node: 5242235029
NVABM TODA terminal node: 6754430756
NVABM TODA terminal node: 7412448639
PCM TODA terminal node: 4712003759
SBTC TODA terminal node: 7679028426
SBTC TODA terminal node: 1437165073
SRMMMMRCDM TODA terminal node: 456858251
SRMMMMRCDM TODA terminal node: 458615236
SRMMMMRCDM TODA terminal node: 7678722356
SRMMMMRCDM TODA terminal node: 373031689
SRMMMMRCDM TODA terminal node: 7673405093
VR TODA terminal node: 678598607


In [ ]:
# get max id

import os, json

def max_osmid(base_path, suffix):
    max_id = 0
    for route in os.listdir(base_path):
        for file in os.listdir(f"{base_path}/{route}"):
            if file.endswith(suffix):
                with open(f"{base_path}/{route}/{file}") as f:
                    data = json.load(f)
                ids = [int(feat['properties']['osmid']) for feat in data['features']]
                if ids:
                    max_id = max(max_id, max(ids))
    return max_id

edgeIdMax = max(
    max_osmid("graph/transit_graph/trike_routes", "edges.geojson"),
    max_osmid("graph/transit_graph/jeep_routes", "edges.geojson")
)

nodeIdMax = max(
    max_osmid("graph/transit_graph/trike_routes", "nodes.geojson"),
    max_osmid("graph/transit_graph/jeep_routes", "nodes.geojson")
)

print(edgeIdMax)
print(nodeIdMax)

16123
14171


In [ ]:
# Add weight for walking edges

jsonToWrite = {
    "type": "FeatureCollection",
    "name": "mabalacatAngelesGraph_edges",
    "crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
    "features": [
    ]
}

with open('graph/original_graph/mabalacatAngeles_edges.geojson', 'r') as file:
    jsonData = json.load(file)


# m/s
walkingSpeed = 1.25

for i in jsonData['features']:
    i['properties']['weight'] = round(round(i['properties']['length'], 2) / walkingSpeed, 2)


    jsonToWrite['features'].append(i)


with open('graph/transit_graph/walking_graph/mabalacatAngelesWalking_edges.geojson', 'w') as file:
    json.dump(jsonToWrite, file)





In [109]:
# Add mode for walking nodes

jsonToWrite = {
    "type": "FeatureCollection",
    "name": "mabalacatAngelesGraph_nodes",
    "crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
    "features": [
    ]
}

with open('graph/original_graph/mabalacatAngeles_nodes.geojson', 'r') as file:
    jsonData = json.load(file)

for i in jsonData['features']:
    i['properties']['mode'] = {
        "type": "walk"
    }

    jsonToWrite['features'].append(i)

with open('graph/transit_graph/walking_graph/mabalacatAngelesWalking_nodes.geojson', 'w') as file:
    json.dump(jsonToWrite, file)


In [3]:
# Loops through every route json 2
import os
import json
from shapely.geometry import shape

relativePath = "graph/transit_graph/jeep_routes"

for route in os.listdir(relativePath):

    for file in os.listdir(f"{relativePath}/{route}"):

        if file.endswith('_nodes.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                nodesJson = json.load(f)

            for feature in nodesJson["features"]:
                props = feature["properties"].copy()
                geom = feature["geometry"]
                node_id = props["osmid"]
                lon, lat = geom["coordinates"]
                props["x"] = lon
                props["y"] = lat

                G.add_node(node_id, **props)

        elif file.endswith('_edges.geojson'):
            with open(f"{relativePath}/{route}/{file}") as f:
                edgesJson = json.load(f)

            for feature in edgesJson["features"]:
                props = feature["properties"]
                geom = feature["geometry"]

                u = props.pop("u")
                v = props.pop("v")

                G.add_edge(
                    u,
                    v,
                    geometry=shape(geom),
                    **props
                )
        else:
            print("Throw not correct format")

In [ ]:
# Find every permutation of shortest path

import osmnx as ox
import geopandas as gpd
from shapely.geometry import LineString
from func import getNearestNodeId

# lat, long
source = [15.13022502299816, 120.58941583884963]
dest = [15.167239872374454, 120.57890970311541]


srcNode = getNearestNodeId(G, source[0], source[1])
destNode = getNearestNodeId(G, dest[0], dest[1])

routesResultsList = []

theList = None
nextG = G.copy()
for i in range(0,3):
    theList = list(ox.k_shortest_paths(nextG, srcNode, destNode, 1, weight="weight"))
    routesResultsList.append(theList[0])

    nextG = nextG.copy()

    routeToRemove = None
    for i in theList[0]:
        if G.nodes[i]['mode']['type'] != 'walk':
            routeToRemove = G.nodes[i]['mode']['details']['name']
            break

    if routeToRemove != None:
        for i in theList[0]:
            if nextG.nodes[i]['mode']['type'] != 'walk':
                if nextG.nodes[i]['mode']['details']['name'] == routeToRemove:
                    nextG.remove_node(i)

    

# Finds all permutations from the list of calculated routes differentiated by modes
permutationList = []
for i in routesResultsList:
    permutation = []
    for j in i:
        if permutation:
            modeType = G.nodes[j]['mode']['type'] if G.nodes[j]['mode']['type'] == 'walk' else G.nodes[j]['mode']['details']['name']
            if modeType != permutation[len(permutation)-1]:
                permutation.append('walk') if G.nodes[j]['mode']['type'] == 'walk' else permutation.append(G.nodes[j]['mode']['details']['name'])
        else:
            permutation.append('walk') if G.nodes[j]['mode']['type'] == 'walk' else permutation.append(G.nodes[j]['mode']['details']['name']) 


    total_weight = sum(G[u][v][0].get('weight', 0) for u, v in zip(i[:-1], i[1:]))
    # print(f"Weight: {total_weight:.2f}: {permutation}")

    if permutation not in permutationList:
        permutationList.append(permutation)

for i in permutationList:
    print(i)


['walk', 'Checkpoint-HolyAngelUniversity-Highway-Angeles', 'walk']
['walk', 'Checkpoint-HolyAngelUniversity', 'walk']
['walk', 'Marisol-Pampang', 'walk']


In [ ]:
# Haversine

import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def nearest_node(G, lat, lon):
    return min(G.nodes, key=lambda n: haversine(lat, lon, G.nodes[n]['y'], G.nodes[n]['x']))

nearest_node = nearest_node(G, 15.158640690032867, 120.59424317411658)

print(nearest_node)

13716443917


: 

In [89]:
# EXPORT SIMPLIFIED GRAPH

import time
import networkx as nx
import osmnx as ox
import json
from contextlib import contextmanager
from func import haversine
                            
@contextmanager
def perfTimer(action=""):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"Elapsed time to {action}: {round(end - start, 2)}s", flush=True)


import os
import json
from shapely.geometry import shape

# Initialize Simplified G
simplifiedG = nx.MultiDiGraph()
relativePath = "graph/transit_graph/jeep_routes"

# Adds every jeep route (nodes/edges) to simplifiedG
with perfTimer('add every jeep route (nodes/edges) to simplifiedG'):
    for route in os.listdir(relativePath):

        for file in os.listdir(f"{relativePath}/{route}"):

            with open(f"{relativePath}/{route}/{route}_nodes.geojson") as f:
                nodesJson = json.load(f)

            for feature in nodesJson["features"]:

                props = feature["properties"].copy()
                geom = feature["geometry"]
                node_id = props["osmid"]
                lon, lat = geom["coordinates"]
                props["x"] = lon
                props["y"] = lat

                if G.nodes[node_id]['mode']['type'] == 'walk':
                    continue

                simplifiedG.add_node(node_id, **props)

            with open(f"{relativePath}/{route}/{route}_edges.geojson") as f:
                edgesJson = json.load(f)

            for feature in edgesJson["features"]:
                props = feature["properties"]
                geom = feature["geometry"]

                u = props.pop("u")
                v = props.pop("v")

                if G.nodes[u]['mode']['type'] == 'walk' or G.nodes[v]['mode']['type'] == 'walk':
                    # print(f'{u}: {G.nodes[u]['mode']['type']} or {v}: {G.nodes[v]['mode']['type']}')
                    continue

                simplifiedG.add_edge(
                    u,
                    v,
                    geometry=shape(geom),
                    **props
                )

for route in os.listdir(relativePath):
    srcNodes = {'northbound': [], 'southbound': []}

    with open(f'{relativePath}/{route}/{route}_nodes.geojson', 'r') as file:
        f = json.load(file)

    # x,y -> node id in route (only used if two routes arent adjacent)
    coord_to_nodeid_route = {}


    # Add route nodes to srcNodes dict
    for i in f['features']:
        coord_to_nodeid_route[
            (i['properties']['x'], i['properties']['y'])
        ] = i['properties']['osmid']
        srcNodes['northbound'].append(i['properties']['osmid']) if i['properties']['mode']['details']['bound'] == 'north' else srcNodes['southbound'].append(i['properties']['osmid'])

    # x,y -> nodeid
    coord_to_nodeid = {(G.nodes[n]['x'], G.nodes[n]['y']): n for n in walkingG.nodes}
    
    # For every route that source route will go to
    for routeDestination in os.listdir(relativePath):
        if route != routeDestination: # makes sure that route does not have an edge going to itself
            
            destNodes = {'northbound': [], 'southbound': []}

            with open(f'graph/transit_graph/jeep_routes/{routeDestination}/{routeDestination}_nodes.geojson', 'r') as file:
                fd = json.load(file)

            for i in fd['features']:    
                coord_to_nodeid_route[
                    (i['properties']['x'], i['properties']['y'])
                ] = i['properties']['osmid']

                destNodes['northbound'].append(i['properties']['osmid']) if i['properties']['mode']['details']['bound'] == 'north' else destNodes['southbound'].append(i['properties']['osmid'])

            # for northbound and southbound src lists
            for srcBound, srcArr in srcNodes.items():

                # for northbound and southbound dest lists
                for destBound, destArr in destNodes.items():
                    
                    # ----- #
                    adjacent = False
                    for srcNode in srcArr:
                            for destNode in destArr:
                                
                                # checks if node in route 1 and node in route 2 are on the same point
                                if G.nodes[srcNode]['y'] == G.nodes[destNode]['y'] and G.nodes[srcNode]['x'] == G.nodes[destNode]['x']:
                                    # Check if your neighbor node doesnt have an adjacent node from the other route, if true, then add a edge from srcNode to destNode using G
                                    for i in list(set(simplifiedG.successors(srcNode)) | set(simplifiedG.predecessors(srcNode))):
                                        listOfAdjacent = [node for node in fd['features'] if node['properties']['x'] == simplifiedG.nodes[i]['x'] and node['properties']['y'] == simplifiedG.nodes[i]['y']]

                                        if not listOfAdjacent:
                                            

                                            # Add walking node where src and dest nodes alight and board on respectively
                                            matchingWalkingNode = coord_to_nodeid[(G.nodes[srcNode]['x'], G.nodes[srcNode]['y'])]
                                            # (clunky fix)
                                            attrs = G.nodes[matchingWalkingNode].copy()
                                            attrs.pop("osmid", None)

                                            simplifiedG.add_node(int(matchingWalkingNode), **attrs)

                                            # Connect source route node -> walking node -> dest route node
                                            edgeAttr = next(iter(G[srcNode][matchingWalkingNode].values()))
                                            simplifiedG.add_edge(srcNode, matchingWalkingNode, **edgeAttr)

                                            edgeAttr = next(iter(G[matchingWalkingNode][destNode].values()))
                                            simplifiedG.add_edge(matchingWalkingNode, destNode, **edgeAttr)

                                            # simplifiedG.add_edge(srcNode, destNode, weight=310) # Please adjust 310 in case inbound and outbound weight changes or do SP from srcNode to destNode
                                            break
                    
                                    adjacent = True

                    if not adjacent:
                        # Gets the closest node

                        # Checks if array from source and destination routes are not empty
                        if not srcArr or not destArr:
                            continue

                        # Gets all walking nodes from source route nodes
                        walkingSource = []
                        for i in srcArr:
                            walkingSource.append(coord_to_nodeid[(G.nodes[i]['x'], G.nodes[i]['y'])])

                        # Gets all walking nodes from dest route nodes
                        walkingDest = []
                        for i in destArr:
                            walkingDest.append(coord_to_nodeid[(G.nodes[i]['x'], G.nodes[i]['y'])])
                            # d.append(ox.nearest_nodes(walkingG, X=G.nodes[i]['x'], Y=G.nodes[i]['y']))

                        # Multi source dijkstra from list of source nodes to list of dest nodes
                        lengths, paths = nx.multi_source_dijkstra(
                            walkingG,
                            sources=walkingSource,
                            weight="weight"
                        )

                        # Get node from destination route that is the shortest by walking from source route
                        closest_b = min(
                            (b for b in walkingDest if b in lengths),
                            key=lengths.get
                        )

                        # Get shortest path from walking source node to walking dest node
                        best_path = paths[closest_b]

                        # Adds walking node from best path into simplified G
                        for osmid in best_path:
                            # Remove osmid first to readd it (clunky fix)
                            attrs = G.nodes[osmid].copy()
                            attrs.pop("osmid", None)
                            simplifiedG.add_node(osmid, **attrs)

                            
                        # Gets the source route's node and append it to the first node of the best path
                        best_path.insert(0, coord_to_nodeid_route[(walkingG.nodes[paths[closest_b][0]]['x'], walkingG.nodes[paths[closest_b][0]]['y'])])

                        # Gets the dest route's best node and append it to the last node of the best path
                        best_path.append(coord_to_nodeid_route[float((walkingG.nodes[closest_b]['x'])), float(walkingG.nodes[closest_b]['y'])])

                        # Adds a connection of best path from route A to B
                        for u, v in zip(best_path, best_path[1:]):
                            attrs = G[u][v][0]
                            simplifiedG.add_edge(u, v, **attrs)

relativePathTrike = "graph/transit_graph/trike_routes"
for subgraph in os.listdir(relativePathTrike):
    terminalList = []
    with open(f"{relativePathTrike}/{subgraph}/{subgraph}_nodes.geojson") as f:
        subGNodesJson = json.load(f)

    # Initialize subgraph G used for finding shortest path within the subgraph's scope
    def loadGraphFromGeojson(nodesPath, edgesPath):
        with open(nodesPath) as f:
            nodesData = json.load(f)

        with open(edgesPath) as f:
            edgesData = json.load(f)

        G = nx.MultiDiGraph()

        for feature in nodesData['features']:
            props = feature['properties']
            osmid = props['osmid']
            G.add_node(osmid, **props)

        for feature in edgesData['features']:
            props = feature['properties']
            u = props['u']
            v = props['v']
            key = props.get('key', 0)

            otherProps = {k: val for k, val in props.items() if k != 'key'}

            G.add_edge(u, v, key=key, geometry=shape(feature['geometry']), **otherProps)

        return G

    subgraphG = loadGraphFromGeojson(
        f"{relativePathTrike}/{subgraph}/{subgraph}_nodes.geojson",
        f"{relativePathTrike}/{subgraph}/{subgraph}_edges.geojson"
    )

    # Add every terminal node data from subgraph
    for i in subGNodesJson['features']:
        if i['properties']['mode']['details']['terminal'] == True:
            terminalList.append(i)

    coord_to_nodeid_walk = {(walkingG.nodes[n]['x'], walkingG.nodes[n]['y']): n for n in walkingG.nodes}
    coord_to_nodeid_subG = {(n['properties']['x'], n['properties']['y']): n['properties']['osmid'] for n in subGNodesJson['features']}

    for route in os.listdir(relativePath):
        destNodes = {'northbound': [], 'southbound': []}
        coord_to_node_route = {}
        with open(f'{relativePath}/{route}/{route}_nodes.geojson', 'r') as file:
            routeNodesJson = json.load(file)

            # Add destination route nodes to destNodes dict
            for i in routeNodesJson['features']:
                coord_to_node_route[
                    (i['properties']['x'], i['properties']['y'])
                ] = i['properties']['osmid']
                destNodes['northbound'].append(i) if i['properties']['mode']['details']['bound'] == 'north' else destNodes['southbound'].append(i)

            # Loop through each bound of route
            for bound, nodes in destNodes.items():

                # Skip if bound has no nodes
                if not nodes:
                    continue
                
                # Finds every adjacent nodes from dest route
                adjacentNodesFromRoute = []
                for i in nodes:
                    if (i['properties']['x'], i['properties']['y']) in list(coord_to_nodeid_subG.keys()):
                        adjacentNodesFromRoute.append(i)

                if adjacentNodesFromRoute:
                    

                    print('---')
                    print(f"{terminal['properties']['osmid']} at {(terminal['properties']['y'], terminal['properties']['x'])}")
                    print(f"going to {route} {bound}")
                    # Get all the nodes from the destination route with its subgraph node equivalent (only adjacent route nodes with subgraph)
                    print('adjacentNodesFromRoute vvv')
                    print([i['properties']['osmid'] for i in adjacentNodesFromRoute])
                    print('---')
                    destNodesOnSubG = [coord_to_nodeid_subG[(n['properties']['x'], n['properties']['y'])] for n in adjacentNodesFromRoute]

                    for terminal in terminalList:
                        # terminalNodeOnWalkingG = coord_to_nodeid_walk[(terminal['properties']['x'], terminal['properties']['y'])]

                        # Get shortest path from trike terminal to jeep route on walking graph
                        print(f"im on {route} {bound} going to {subgraph} {(terminal['properties']['y'], terminal['properties']['x'])}")

                        print(destNodesOnSubG)
                        lengths, paths = nx.single_source_dijkstra(subgraphG, source=terminal['properties']['osmid'], weight="weight")
                        reachable = [n for n in destNodesOnSubG if n in lengths]
                        closest = min(reachable, key=lengths.get)
                        best_path = paths[closest]

                        # Add nodes from trike terminal to best node
                        shortestPathOnSubG = best_path
                        # for i in best_path:
                        #     # Get node ids from subgraph from walking nodes
                        #     shortestPathOnSubG.append(coord_to_nodeid_subG[(walkingG.nodes[i]['x'], walkingG.nodes[i]['y'])])

                        # Add walking node where trike node alights from and where it boards jeep node on
                        shortestPathOnSubG.append(coord_to_nodeid_walk[(subgraphG.nodes[closest]['x'], subgraphG.nodes[closest]['y'])])
                        shortestPathOnSubG.append(coord_to_node_route[(G.nodes[closest]['x'], G.nodes[closest]['y'])])

                        
                        # Add shortest path nodes to simplified graph
                        for osmid in shortestPathOnSubG:
                            if osmid not in simplifiedG:
                                # Remove osmid first to readd it (clunky fix)
                                attrs = G.nodes[osmid].copy()
                                attrs.pop("osmid", None)
                                simplifiedG.add_node(osmid, **attrs)

                        # Connect shortest path nodes to simplified graph
                        for u, v in zip(shortestPathOnSubG, shortestPathOnSubG[1:]):
                            attrs = G[u][v][0]
                            simplifiedG.add_edge(u, v, **attrs)

                routeNodeIds = [i for i in nodes]
                routeNodeIdsWalk = [coord_to_nodeid_walk[(n['properties']['x'], n['properties']['y'])] for n in routeNodeIds]
                # print(routeNodeIdsWalk)

                # Add path from xbound route to every terminal in a subgraph
                for terminal in terminalList:
                    terminalNodeOnWalkingG = coord_to_nodeid_walk[(terminal['properties']['x'], terminal['properties']['y'])]

                    lengths, paths = nx.multi_source_dijkstra(
                        walkingG,
                        sources=routeNodeIdsWalk,
                        target=terminalNodeOnWalkingG,
                        weight="weight"
                    )

                    pathFromRouteToTerminal = paths

                    # Find the route node before the shortest path on walking graph
                    firstNodeOnPath = None
                    for i in routeNodeIds:
                        if i['properties']['x'] == walkingG.nodes[paths[0]]['x'] and i['properties']['y'] == walkingG.nodes[paths[0]]['y']:
                            firstNodeOnPath = i['properties']['osmid']

                    # First node of path
                    pathFromRouteToTerminal.insert(0, firstNodeOnPath)
                    pathFromRouteToTerminal.append(terminal['properties']['osmid'])

                    # Add shortest path nodes to simplifiedG
                    for osmid in pathFromRouteToTerminal:
                        # Remove osmid first to readd it (clunky fix)
                        attrs = G.nodes[osmid].copy()
                        simplifiedG.add_node(osmid, **attrs)

                    # Add shortest path edges to simplifiedG

                    print('----')
                    print(pathFromRouteToTerminal)
                    for u, v in zip(pathFromRouteToTerminal, pathFromRouteToTerminal[1:]):
                        print(f"{u} and {v}")
                        attrs = G[u][v][0]
                        simplifiedG.add_edge(u, v, **attrs)


simplifiedG.graph['crs'] = 'epsg:4326'
nodes, edges = ox.graph_to_gdfs(simplifiedG)
nodes = nodes.drop(columns="osmid")
nodes.to_file("graph/transit_graph/simplified_graph/simplifiedGraph_nodes.geojson", driver='GeoJSON')
edges.to_file("graph/transit_graph/simplified_graph/simplifiedGraph_edges.geojson", driver='GeoJSON')


Elapsed time to add every jeep route (nodes/edges) to simplifiedG: 0.27s
----
[1832, 6514686662, 372563684, 8671620943, 7198793859, 10095064735, 494626553, 7198793869, 7198818594, 10094834292, 619846077, 465625250, 311719355, 7198793874, 7719869363, 7719869365, 311719352, 7686499460, 2001392114, 13474271329, 13474271322, 619846047, 2001474788, 13474271332, 457888892, 1118736623, 7678710918, 7676444992, 1118736621, 7676426694, 8693348902, 12213572036, 508031777, 457892864, 7676485439, 457890693, 11014300822, 9870731149, 9870731141, 9870731163, 9870731156, 1190973691, 7677738752, 7677738558, 11014250266, 2102898846, 7676205369, 13474271342, 13474271348, 7677738554, 7676208419, 13476553254, 13476553263, 2102900822, 8679646651, 7677708856, 7677720806, 9870731122, 9870731131, 6125989052, 9910238144, 1214489111, 10118861219, 10118861224, 2102905523, 2102905377, 8411355077, 2102905037, 7676750699, 1214488897, 1214488550, 2102909025, 2102910290, 7676750696, 8407211239, 313121427, 7676409378, 7

In [81]:

inbound = list(G.predecessors(7678722356 ))
outbound = list(G.successors(7678722356 ))

print(inbound)   # ['A', 'C']
print(outbound)  # ['D', 'E']

[2780788658, 7678722355, 8682232784, 3298]
[2780788658, 8682232784, 3298]


In [53]:
import osmnx as ox

simplifiedG = nx.MultiDiGraph()
relativePathTrike = "graph/transit_graph/trike_routes"
relativePath = "graph/transit_graph/jeep_routes"

terminalList = []
for subgraph in os.listdir(relativePathTrike):
    if subgraph == "Zone 1":
            

        with open(f"{relativePathTrike}/{subgraph}/{subgraph}_nodes.geojson") as f:
            subGNodesJson = json.load(f)

            # Add every terminal node data from subgraph
            for i in subGNodesJson['features']:
                if i['properties']['mode']['details']['terminal'] == True:
                    terminalList.append(i)

        # x,y -> node id in route (only used if two routes arent adjacent)
        coord_to_nodeid_walk = {(G.nodes[n]['x'], G.nodes[n]['y']): n for n in walkingG.nodes}
        coord_to_nodeid_subG = {(n['properties']['x'], n['properties']['y']): n['properties']['osmid'] for n in subGNodesJson['features']}

        

        for route in os.listdir(relativePath):
            destNodes = {'northbound': [], 'southbound': []}
            coord_to_node_route = {}
            with open(f'{relativePath}/{route}/{route}_nodes.geojson', 'r') as file:
                routeNodesJson = json.load(file)


                # Add destination route nodes to destNodes dict
                for i in routeNodesJson['features']:
                    coord_to_node_route[
                        (i['properties']['x'], i['properties']['y'])
                    ] = i['properties']['osmid']
                    destNodes['northbound'].append(i) if i['properties']['mode']['details']['bound'] == 'north' else destNodes['southbound'].append(i)

                # Loop through each bound of route
                for bound, nodes in destNodes.items():

                    # Skip if bound has no nodes
                    if not nodes:
                        continue
                    
                    # Finds every adjacent nodes from dest route
                    adjacentNodes = []
                    for i in subGNodesJson['features']:
                        if (i['properties']['x'], i['properties']['y']) in list(coord_to_node_route.keys()):
                            adjacentNodes.append(i['properties']['osmid'])
                    
                    
                    if adjacentNodes:

                        # Get all the nodes from the destination route with its walking node equivalent
                        destNodesOnWalkG = [coord_to_nodeid_walk[(n['properties']['x'], n['properties']['y'])] for n in nodes]

                        for terminal in terminalList:
                            terminalNodeOnWalkingG = coord_to_nodeid_walk[(terminal['properties']['x'], terminal['properties']['y'])]

                            # Get shortest path from trike terminal to jeep route on walking graph
                            lengths, paths = nx.single_source_dijkstra(walkingG, source=terminalNodeOnWalkingG, weight="weight")
                            closest = min(destNodesOnWalkG, key=lengths.get)
                            best_path = paths[closest]

                            # Add nodes from trike terminal to jeep route
                            shortestPathOnSubG = []
                            for i in best_path:
                                # Get node ids from subgraph from walking nodes
                                shortestPathOnSubG.append(coord_to_nodeid_subG[(walkingG.nodes[i]['x'], walkingG.nodes[i]['y'])])

                            # Add walking node where trike node alights from and where it boards jeep node on
                            shortestPathOnSubG.append(coord_to_nodeid_walk[(walkingG.nodes[closest]['x'], walkingG.nodes[closest]['y'])])
                            shortestPathOnSubG.append(coord_to_node_route[(walkingG.nodes[closest]['x'], walkingG.nodes[closest]['y'])])

                            print('---')
                            print("shortestPathOnSubG")
                            print(shortestPathOnSubG)
                            print('---')

                            print('///')
                            print("best_path")
                            print(best_path)
                            print('///')
                            # Add shortest path nodes to simplified graph
                            for osmid in shortestPathOnSubG:
                                if osmid not in simplifiedG:
                                    # Remove osmid first to readd it (clunky fix)
                                    attrs = G.nodes[osmid].copy()
                                    attrs.pop("osmid", None)
                                    simplifiedG.add_node(osmid, **attrs)

                            # Connect shortest path nodes to simplified graph
                            for u, v in zip(shortestPathOnSubG, shortestPathOnSubG[1:]):
                                attrs = G[u][v][0]
                                simplifiedG.add_edge(u, v, **attrs)
                    else:
                        # That means if subgraph dont have any adjacent nodes with any routes, its practically useless
                        print()


                    routeNodeIds = [i for i in nodes]
                    routeNodeIdsWalk = [coord_to_nodeid_walk[(n['properties']['x'], n['properties']['y'])] for n in routeNodeIds]
                    # print(routeNodeIdsWalk)

                    # Add path from xbound route to every terminal in a subgraph
                    for terminal in terminalList:
                        terminalNodeOnWalkingG = coord_to_nodeid_walk[(terminal['properties']['x'], terminal['properties']['y'])]

                        lengths, paths = nx.multi_source_dijkstra(
                            walkingG,
                            sources=routeNodeIdsWalk,
                            target=terminalNodeOnWalkingG,
                            weight="weight"
                        )

                        pathFromRouteToTerminal = paths

                        # Find the route node before the shortest path on walking graph
                        firstNodeOnPath = None
                        for i in routeNodeIds:
                            if i['properties']['x'] == walkingG.nodes[paths[0]]['x'] and i['properties']['y'] == walkingG.nodes[paths[0]]['y']:
                                firstNodeOnPath = i['properties']['osmid']

                        # First node of path
                        pathFromRouteToTerminal.insert(0, firstNodeOnPath)
                        pathFromRouteToTerminal.append(terminal['properties']['osmid'])

                        # Add shortest path nodes to simplifiedG
                        for osmid in pathFromRouteToTerminal:
                            # Remove osmid first to readd it (clunky fix)
                            attrs = G.nodes[osmid].copy()
                            attrs.pop("osmid", None)
                            simplifiedG.add_node(osmid, **attrs)

                        

                        # Add shortest path edges to simplifiedG
                        for u, v in zip(pathFromRouteToTerminal, pathFromRouteToTerminal[1:]):
                            attrs = G[u][v][0]
                            simplifiedG.add_edge(u, v, **attrs)

                    


                    





---
shortestPathOnSubG
[3836, 6627, 4814, 4099, 6624, 6625, 4792, 5867, 3692, 6043, 6042, 5327, 3693, 3931, 4100, 6389, 5330, 5324, 3961, 6398, 5320, 6181, 3814, 5155, 6514686664, 1833]
---
///
best_path
[457888892, 13474271332, 2001474788, 619846047, 13474271322, 13474271329, 2001392114, 7686499460, 311719352, 7719869365, 7719869363, 7198793874, 311719355, 465625250, 619846077, 10094834292, 7198818594, 7198793869, 494626553, 10095064735, 7198793859, 8671620943, 372563684, 6514686664]
///
---
shortestPathOnSubG
[4046, 3721, 3723, 6289, 6288, 3724, 3725, 6291, 3726, 3753, 3687, 3754, 3828, 3827, 3823, 3796, 3752, 3804, 6400, 3689, 3690, 4088, 4083, 3783, 6699, 3934, 3993, 3933, 5722, 5740, 5721, 5739, 3962, 6281, 6285, 5775, 4984, 6266, 5102, 4656, 5715, 5773, 4983, 6578, 6591, 6600, 6599, 6584, 6735, 6596, 6593, 6611, 6707, 6585, 3959, 3957, 3952, 3953, 4842, 4849, 4848, 4837, 2186718334, 1843]
---
///
best_path
[518296272, 314889942, 314890265, 9174863874, 9174863863, 314890303, 31489

[2780788658, 7678722355, 8682232784, 3298]
[2780788658, 8682232784, 3298]


In [5]:
# Adding source and destination nodes to simplified G (Run the previous notebook before this)
import copy
import geopandas as gpd

with perfTimer("read nodes and edges of mabalacat+angeles City"):
    nodes = gpd.read_file("graph/transit_graph/simplified_graph/simplifiedGraph_nodes.geojson")
    edges = gpd.read_file("graph/transit_graph/simplified_graph/simplifiedGraph_edges.geojson")


with perfTimer("Fix index of nodes and edges"):
    nodes = nodes.set_index("osmid")
    edges = edges.set_index(["u", "v", "key"])


# with perfTimer("convert np arrays to lists"):
#     for attr in edges.columns:
#         new_values = []

#         for val in edges[attr]:
#             if isinstance(val, np.ndarray):
#                 new_values.append(val.tolist())
#             else:
#                 new_values.append(val)

#         edges[attr] = new_values

with perfTimer("initialize MultiDiGraph"):
    simplifiedGCopy = ox.graph_from_gdfs(nodes, edges)


# with perfTimer('deepcopy'):
#     simplifiedGCopy = copy.deepcopy(simplifiedG)

srcNodeUser = 7818467924
destNodeUser = 10911198627

with perfTimer('add src and dest to G'):
    node_data = G.nodes[srcNodeUser]
    simplifiedGCopy.add_node(srcNodeUser, **node_data)

    node_data = G.nodes[destNodeUser]
    simplifiedGCopy.add_node(destNodeUser, **node_data)


# Loop through each route for srcNodeUser to find its shortest path
for route in os.listdir(relativePath):
    print('------------------')
    with perfTimer(f'route {route}:'):

        routeNodes = {'northbound': {}, 'southbound': {}}

        with open(f'{relativePath}/{route}/{route}_nodes.geojson', 'r') as file:
            f = json.load(file)


        for i in f['features']:
            if i['properties']['mode']['details']['bound'] == 'north':
                routeNodes['northbound'][i['properties']['osmid']] = {
                    'x': i['properties']['x'],
                    'y': i['properties']['y']
                }
            else:
                routeNodes['southbound'][i['properties']['osmid']] = {
                    'x': i['properties']['x'],
                    'y': i['properties']['y']
                }
                

        # route (northbound, southbound) / routeArr {900: {'x': 1.2, 'y': 3.4}}
        for route, routeArr in routeNodes.items():
            if routeArr:
                # list of equivalent nodes on the walking graph
                routeOnWalkingG = []

                with perfTimer('1'):
                    for key, value in routeArr.items():
                        for node, data in G.nodes(data=True):
                            if data['x'] == value['x'] and data['y'] == value['y']:
                                routeOnWalkingG.append(node)
                                break

                # From user source node to route
                with perfTimer('Source node to route'):
                    distance, path = nx.single_source_dijkstra(walkingG, source=srcNodeUser, weight='weight')

    

                # Gets the best equivalent route node with on the walking graph
                best_node = min(routeOnWalkingG, key=lambda n: distance.get(n, float('inf')))

                with perfTimer('2'):
                    # Looks for its route node counterpart
                    for key, value in routeArr.items():
                        if value['x'] == walkingG.nodes[best_node]['x'] and value['y'] == walkingG.nodes[best_node]['y']:
                            bestRouteNode = key
                            break


                simplifiedGCopy.add_edge(path[best_node][0], bestRouteNode, weight=distance[best_node])

                # From route to user destination node
                with perfTimer("Closest edge from route to node"):
                    distance, path = nx.multi_source_dijkstra(walkingG, sources=set(routeOnWalkingG), target=destNodeUser, weight='weight')
                best_node = path[0]

                with perfTimer('3'):
                    for key, value in routeArr.items():
                        if value['x'] == walkingG.nodes[best_node]['x'] and value['y'] == walkingG.nodes[best_node]['y']:
                            bestRouteNode = key
                            break
                
                simplifiedGCopy.add_edge(bestRouteNode, destNodeUser, weight=distance)








    

DataSourceError: graph/transit_graph/simplified_graph/simplifiedGraph_nodes.geojson: No such file or directory

In [35]:
simplifiedGCopy.graph['crs'] = 'epsg:4326'
nodes, edges = ox.graph_to_gdfs(simplifiedGCopy)
nodes = nodes.reset_index(drop=True)

nodes.to_file('nodes.geojson', driver='GeoJSON')
edges.to_file('edges.geojson', driver='GeoJSON')
nodes.to_file('graph/test_graph/simplifiedGCopy_nodes.geojson', driver='GeoJSON')
edges.to_file('graph/test_graph/simplifiedGCopy_edges.geojson', driver='GeoJSON')

In [ ]:
# Finds all permutations from the list of calculated routes differentiated by modes

import osmnx as ox

theList = list(ox.k_shortest_paths(simplifiedGCopy, srcNodeUser, destNodeUser, 5, weight="weight"))

permutationList = []
for i in theList:
    permutation = []

    for j in i:

        if permutation:
            modeType = simplifiedGCopy.nodes[j]['mode']['type'] if simplifiedGCopy.nodes[j]['mode']['type'] == 'walk' else simplifiedGCopy.nodes[j]['mode']['details']['name']
            if modeType != permutation[len(permutation)-1]: # Check if the node before is the same mode
                permutation.append('walk') if simplifiedGCopy.nodes[j]['mode']['type'] == 'walk' else permutation.append(simplifiedGCopy.nodes[j]['mode']['details']['name'])
        else: # First node in the calculated path
            permutation.append('walk') if simplifiedGCopy.nodes[j]['mode']['type'] == 'walk' else permutation.append(simplifiedGCopy.nodes[j]['mode']['details']['name']) 


    total_weight = sum(simplifiedGCopy[u][v][0].get('weight', 0) for u, v in zip(i[:-1], i[1:]))
    # print(f"Weight: {total_weight:.2f}: {permutation}")

    if permutation not in permutationList:
        print(i)
        permutationList.append(permutation)

for i in permutationList:
    print(i)


[9191058305, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 327, 326, 325, 324, 323, 322, 321, 320, 319, 318, 317, 316, 315, 314, 313, 312, 311, 310, 309, 308, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 503, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 81

In [ ]:
import osmnx as ox

theList = list(ox.k_shortest_paths(simplifiedGCopy, srcNodeUser, destNodeUser, 5, weight="weight"))



# Finds all permutations from the list of calculated routes differentiated by modes
permutationList = []
for i in theList:
    permutation = []
    route = i.copy()

    for j in range(len(i)):

        if permutation:
            modeType = simplifiedGCopy.nodes[i[j]]['mode']['type'] if simplifiedGCopy.nodes[i[j]]['mode']['type'] == 'walk' else simplifiedGCopy.nodes[i[j]]['mode']['details']['name']

            if modeType != permutation[-1]:  # Check if the previous mode is the same as the current node's mode
                extended = ox.shortest_path(G, i[j-1], i[j], weight="weight")
                if len(extended) > 2:
                    route[j:j] = extended[1:-1]

                permutation.append('walk') if simplifiedGCopy.nodes[i[j]]['mode']['type'] == 'walk' else permutation.append(
                    simplifiedGCopy.nodes[i[j]]['mode']['details']['name']
                )

        else:  # First node in the calculated path
            permutation.append('walk') if simplifiedGCopy.nodes[i[j]]['mode']['type'] == 'walk' else permutation.append(
                simplifiedGCopy.nodes[i[j]]['mode']['details']['name']
            )


    total_weight = sum(simplifiedGCopy[u][v][0].get('weight', 0) for u, v in zip(i[:-1], i[1:]))
    # print(f"Weight: {total_weight:.2f}: {permutation}")

    if permutation not in permutationList:
        print(f"1: {i}")
        print(f"2: {route}")
        permutationList.append(permutation)

for i in permutationList:
    print(i)



# The issues lie in you should USE THE WALKING GRAPH !!!



1: [7818467924, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 420, 421, 422, 423, 424, 425, 426, 427, 428, 429, 430, 431, 432, 433, 434, 435, 436, 437, 438, 439, 440, 441, 442, 443, 444, 445, 446, 447, 448, 449, 450, 451, 452, 453, 454, 455, 456, 457, 458, 459, 460, 461, 462, 463, 464, 465, 466, 467, 468, 469, 470, 471, 472, 327, 326, 325, 324, 323, 322, 321, 320, 319, 318, 317, 316, 315, 314, 313, 312, 311, 310, 309, 308, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 739, 740, 741, 742, 743, 744, 745, 746, 747, 748, 749, 750, 751, 752, 753, 754, 755, 756, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766, 767, 768, 769, 770, 771, 772, 773, 774, 775, 776, 777, 778, 779, 780, 781, 782, 783, 784, 785, 786, 787, 788, 789, 790, 791, 792, 793, 794, 795, 796, 797, 798, 799, 800, 801, 802, 803, 804, 805, 806, 807, 808, 809, 810, 811, 812, 813, 814, 815, 816, 817,

In [44]:
simplifiedGCopy.graph['crs'] = 'epsg:4326'
nodes, edges = ox.graph_to_gdfs(simplifiedGCopy)
nodes = nodes.reset_index(drop=True)

nodes.to_file('nodes.geojson', driver='GeoJSON')
edges.to_file('edges.geojson', driver='GeoJSON')
nodes.to_file('graph/test_graph/simplifiedGCopy_nodes.geojson', driver='GeoJSON')
edges.to_file('graph/test_graph/simplifiedGCopy_edges.geojson', driver='GeoJSON')

In [24]:
simplifiedG.graph['crs'] = 'epsg:4326'
nodes, edges = ox.graph_to_gdfs(simplifiedG)
nodes = nodes.reset_index(drop=True)
nodes.to_file("graph/test_graph/simplifiedG_nodes.geojson", driver='GeoJSON')
edges.to_file("graph/test_graph/simplifiedG_edges.geojson", driver='GeoJSON')

In [19]:
import math


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def getNearestNodeId(G, lat, lon):
    return min(G.nodes, key=lambda n: haversine(lat, lon, G.nodes[n]['y'], G.nodes[n]['x']))


def getFare(totalDist, mode="walk", discount=False):
    '''
    Jeep fare computation based on LTFRB Fare Matrix for PUJs dated Oct. 8 2023
    Tricycle fare based on Angeles City TODA Fare Matrix (Ordinance No. 723, S-2024)
    '''
    baseJeepFare = 13
    perKmFareIncrease = 1.8
    # totalDist = 0

    if mode == "walk":
        return 0
    
    # for u, v in zip(modePathList, modePathList[1:]):
    #     totalDist += haversine(u[0], u[1], v[0], v[1])

    if mode == "jeep":
        if totalDist < 5:
            if discount:
                return round(baseJeepFare * 0.8 * 4) / 4
            return baseJeepFare
        else:
            fare = baseJeepFare + (perKmFareIncrease * (math.floor(totalDist) - 4))
            if discount:
                return round(fare * 0.8 * 4) / 4
            return round(fare * 4) / 4
    elif mode == "trike":
        if totalDist < 1.1:
            if discount:
                return 35 * 0.8
            return 35
        else:
            increments = int(totalDist - 0.1)
            fare = 35 + max(0, increments) * 15
            if discount:
                return fare * 0.8
            return fare

             

getFare(6.1, mode="trike", discount=True)




100.0